# Opportunity Scout AI — Data Pipeline

**Architecture:**
```
User Input (industry, country, year)
        ↓
AggregationEngine
        ↓
Live Collectors:                    Fallback:
- GoogleTrendsCollector             VenturesCollector (ventures.csv)
- WorldBankCollector
- RedditCollector
- GDELTCollector          ← NEW
        ↓
Rate Limiter (per-collector)        ← NEW
        ↓
Signal Validation
        ↓
Normalization (stored min/max)
        ↓
Opportunity Score Engine
        ↓
Data Quality Scorer                 ← NEW
        ↓
Output JSON  ← mode: 'live' | 'historical'
              data_quality: 0.0-1.0 ← NEW
```

**Modes:**
- `Mode A (historical)` — batch-scores rows from ventures.csv (used by ML team for training)
- `Mode B (live)` — scores a new user query using live collectors + CSV fallback

#### Imports

In [1]:
import pandas as pd
import numpy as np
import requests
import datetime
import time
import unittest
import threading
from abc import ABC, abstractmethod
from collections import deque


!pip install --upgrade google-cloud-bigquery
!pip install pytrends

Requirement already up-to-date: google-cloud-bigquery in c:\users\hp\anaconda3\envs\learn-env\lib\site-packages (3.30.0)


#### Load Dataset & Explore

In [2]:
df = pd.read_csv("../Data/ventures.csv")

print(df.columns.tolist())
print(df.head())
print(df.info())
print(df.describe())

['name', 'industry', 'category', 'founded_year', 'country', 'status', 'funding_total_usd', 'funding_rounds', 'opportunity_score', 'opportunity_category', 'trend_slope', 'news_volume', 'news_sentiment', 'reddit_density', 'gdp_growth', 'inflation', 'startup_density', 'log_funding']
                 name    industry               category  founded_year  \
0            #waywire        News  Media & Entertainment          2012   
1    Rock' Your Paper  Publishing  Media & Entertainment          2012   
2  -R- Ranch and Mine     Tourism   Travel & Hospitality          2014   
3    004 Technologies    Software  Technology & Software          2010   
4         1,2,3 Listo  E-Commerce    E-Commerce & Retail          2012   

         country     status  funding_total_usd  funding_rounds  \
0  United States   acquired            1750000               1   
1        Estonia  operating              40000               1   
2  United States  operating              60000               2   
3  United 

#### Schema & Feature Bounds

These bounds are derived from ventures.csv and saved to disk.
The ML team and API team will reference `feature_bounds.csv` for consistent normalization.

In [3]:
# Core signals the pipeline must produce
CORE_SCHEMA = {
    "trend_slope":      float,
    "news_volume":      float,
    "news_sentiment":   float,
    "reddit_density":   float,
    "gdp_growth":       float,
    "inflation":        float,
    "startup_density":  float,
}

# Features used in scoring + normalization
SCORE_FEATURES = [
    "trend_slope",
    "gdp_growth",
    "reddit_density",
    "news_sentiment",
    "news_volume",
    "startup_density",
    "log_funding",
    "inflation",
]

# Features the validator checks for in every output
REQUIRED_FEATURES = [
    "trend_slope",
    "gdp_growth",
    "reddit_density",
    "news_sentiment",
    "news_volume",
    "startup_density",
    "funding_total_usd",  # raw — log-transformed inside scorer
    "inflation",
]

# Which features are considered 'live' (sourced from live collectors, not CSV)
# Used by the data quality scorer to determine how fresh the signals are
LIVE_FEATURES = [
    "trend_slope",     # GoogleTrends
    "gdp_growth",      # WorldBank
    "inflation",       # WorldBank
    "reddit_density",  # Reddit
    "news_volume",     # GDELT
    "news_sentiment",  # GDELT
]

# Compute and save feature bounds from the dataset
bounds_list = []
for feature in SCORE_FEATURES:
    bounds_list.append({
        "feature": feature,
        "min":     df[feature].min(),
        "max":     df[feature].max(),
        "median":  df[feature].median(),
    })

feature_bounds_df = pd.DataFrame(bounds_list).set_index("feature")
feature_bounds_df.to_csv("feature_bounds.csv")
print("Feature bounds saved.")
feature_bounds_df

Feature bounds saved.


,min,max,median
feature,,,
trend_slope,-0.231172,0.364155,-0.055052
gdp_growth,-16.040020,14.519750,2.117830
reddit_density,0.000000,280.337400,0.702800
news_sentiment,-0.011100,6.592600,6.415900
news_volume,9041.000000,989564.000000,149705.000000
startup_density,0.003000,18.484600,5.092400
log_funding,0.000000,9.414973,5.989107
inflation,-4.447547,53.228698,2.138384


#### Logging & In-Memory Cache

In [4]:
# ── Logging ──────────────────────────────────────────────────────────────────

def log_event(level, message):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] [{level}] {message}")


# ── In-Memory Cache ───────────────────────────────────────────────────────────
# NOTE: For production, swap CACHE dict with Redis (see project plan Phase 1.2)

CACHE = {}

def get_cache_key(industry, country, year):
    return f"{industry}|{country}|{year}"

def get_cached(key):
    entry = CACHE.get(key)
    if entry:
        log_event("INFO", f"Cache hit for key: {key}")
    return entry

def set_cache(key, value):
    CACHE[key] = value
    log_event("INFO", f"Cached result for key: {key}")

def clear_cache():
    CACHE.clear()
    log_event("INFO", "Cache cleared")

#### Rate Limiter

A sliding-window rate limiter used to wrap every collector call.
Each collector gets its own `RateLimiter` instance with its own call budget.
This prevents hitting API rate limits and getting blocked.

| Collector       | Default limit       | Why                                       |
|-----------------|---------------------|-------------------------------------------|
| GoogleTrends    | 5 calls / 60 sec    | Unofficial API — very aggressive blocking |
| WorldBank       | 10 calls / 60 sec   | Free public API, generous but be polite   |
| Reddit          | 10 calls / 60 sec   | Public JSON, no auth = strict limits      |
| GDELT           | 5 calls / 60 sec    | Free API, but penalises rapid queries     |

In [5]:
class RateLimiter:
    """
    Sliding-window rate limiter.
    Tracks timestamps of recent calls in a deque.
    If the budget is exhausted, blocks (sleeps) until the oldest call
    falls outside the window.

    Usage:
        limiter = RateLimiter(max_calls=5, period_seconds=60)
        limiter.wait()   # call this before every API request
    """

    def __init__(self, max_calls: int, period_seconds: float):
        self.max_calls   = max_calls
        self.period      = period_seconds
        self._call_times = deque()
        self._lock       = threading.Lock()

    def wait(self):
        """Block if rate limit would be exceeded, then register this call."""
        with self._lock:
            now = time.monotonic()

            # Drop timestamps older than the window
            while self._call_times and now - self._call_times[0] > self.period:
                self._call_times.popleft()

            if len(self._call_times) >= self.max_calls:
                # Wait until oldest call expires from the window
                sleep_for = self.period - (now - self._call_times[0])
                if sleep_for > 0:
                    log_event("INFO", f"RateLimiter: sleeping {sleep_for:.1f}s to respect limit")
                    time.sleep(sleep_for)

                # Re-prune after sleep
                now = time.monotonic()
                while self._call_times and now - self._call_times[0] > self.period:
                    self._call_times.popleft()

            self._call_times.append(time.monotonic())

    def remaining(self) -> int:
        """How many calls are left in the current window."""
        now    = time.monotonic()
        active = sum(1 for t in self._call_times if now - t <= self.period)
        return max(0, self.max_calls - active)


# ── Shared limiter instances (one per collector) ──────────────────────────────
RATE_LIMITERS = {
    "GoogleTrends": RateLimiter(max_calls=5,  period_seconds=60),
    "WorldBank":    RateLimiter(max_calls=10, period_seconds=60),
    "Reddit":       RateLimiter(max_calls=10, period_seconds=60),
    "GDELT":        RateLimiter(max_calls=5,  period_seconds=60),
}

#### Signal Validation

In [6]:
def validate_signals(signals):
    """
    Checks all REQUIRED_FEATURES are present, numeric, and not NaN.
    Returns:
        validated (dict) — cleaned signals
        issues    (dict) — feature -> issue description
    """
    validated = {}
    issues    = {}

    for feature in REQUIRED_FEATURES:

        if feature not in signals:
            validated[feature] = np.nan
            issues[feature]    = "missing_key"
            continue

        value = signals[feature]

        try:
            validated[feature] = float(value)
            if np.isnan(validated[feature]):
                issues[feature] = "nan_value"
        except (TypeError, ValueError):
            validated[feature] = np.nan
            issues[feature]    = "type_error"

    # Pass-through any extra signals
    for k, v in signals.items():
        if k not in validated:
            try:
                validated[k] = float(v)
            except (TypeError, ValueError):
                pass

    if issues:
        log_event("WARN", f"Signal issues detected: {issues}")

    return validated, issues

#### Data Quality Scorer

After validation, we score the *quality* of the signals collected.
Quality is a float from 0.0 to 1.0 and is included in every output JSON.

**How it works:**
- Each of the 6 `LIVE_FEATURES` is worth a fraction of the total score.
- If the signal came from a live collector → full credit (1.0)
- If it came from the CSV fallback → half credit (0.5) — it's real but historical
- If it is NaN or missing entirely → no credit (0.0)

**Quality label thresholds:**
- `High`   ≥ 0.75  — most signals are from live APIs, scores are trustworthy
- `Medium` ≥ 0.50  — mix of live and fallback, treat score with some caution
- `Low`    < 0.50  — mostly fallback or imputed, low confidence in the score

The ML team can use `data_quality` as a sample weight during training.
The frontend should display the `quality_label` badge next to the score.

In [7]:
def score_data_quality(validated_signals: dict, live_signals: dict) -> dict:
    """
    Measures how much of the data came from live sources vs fallback vs missing.

    Args:
        validated_signals : output of validate_signals() — all signals present
        live_signals      : signals dict assembled ONLY from live collectors
                            (before fallback was applied)

    Returns dict with:
        data_quality       : float 0.0 - 1.0
        quality_label      : 'High' | 'Medium' | 'Low'
        quality_breakdown  : per-feature source label
    """
    total_features = len(LIVE_FEATURES)
    score_sum      = 0.0
    breakdown      = {}

    for feature in LIVE_FEATURES:
        value = validated_signals.get(feature)

        # Missing or NaN
        if value is None or (isinstance(value, float) and np.isnan(value)):
            breakdown[feature] = "missing"
            score_sum += 0.0

        # Came from live collector
        elif feature in live_signals and live_signals[feature] is not None:
            breakdown[feature] = "live"
            score_sum += 1.0

        # Came from CSV fallback
        else:
            breakdown[feature] = "fallback"
            score_sum += 0.5

    quality_score = round(score_sum / total_features, 4)

    if quality_score >= 0.75:
        label = "High"
    elif quality_score >= 0.50:
        label = "Medium"
    else:
        label = "Low"

    log_event("INFO", f"Data quality: {quality_score} ({label}) | breakdown: {breakdown}")

    return {
        "data_quality":      quality_score,
        "quality_label":     label,
        "quality_breakdown": breakdown,
    }

---

#### GDP Context Engine

Adds a `gdp_context` field to every pipeline output explaining **why** GDP moved in a given country and year.

**How it works (hybrid approach):**
1. **Static lookup table** — covers every significant GDP event in the dataset (2005–2018).
   Handwritten one-sentence explanations for all extreme and notable movements.
2. **Anthropic API fallback** — for any country+year not in the table, calls Claude to
   generate a one-sentence explanation on the fly.
3. **Graceful degradation** — if the API is unavailable, returns a plain data-only sentence
   (`'GDP was X% in [country] [year].'`) so the output never breaks.

**Output lives in two places:**
- `result['gdp_context']` — in every `run_live()` JSON output
- `gdp_context` column — added to the `run_historical()` DataFrame

**For investors and entrepreneurs:** this single sentence gives immediate market context —
whether a startup launched into a crisis, a boom, or a recovery year.

In [8]:
# ═══════════════════════════════════════════════════════════════════════════
# GDP Context Engine
# Explains WHY a country's GDP rose or fell in a given year.
# Used by investors, entrepreneurs and stakeholders to understand the
# economic backdrop behind a startup's founding conditions.
# ═══════════════════════════════════════════════════════════════════════════

# ── Static lookup table ────────────────────────────────────────────────────
# Keys are (country, year) tuples. Values are one-sentence explanations.
# Coverage: all 65 HIGH-priority events + key MED-priority events from the
# dataset, plus major global events that affect multiple countries.

GDP_CONTEXT_TABLE = {

    # ── 2008–2009 GLOBAL FINANCIAL CRISIS (affects nearly every country) ──
    # Baltic states: worst-hit in Europe
    ("Latvia",        2009): "Latvia's GDP collapsed by 16% as the global financial crisis burst a credit-fuelled property bubble, triggering the worst recession in its post-Soviet history.",
    ("Lithuania",     2009): "Lithuania's GDP fell nearly 15% as the global financial crisis devastated export demand and a domestic credit bubble collapsed simultaneously.",
    ("Estonia",       2009): "Estonia's GDP dropped 14.6% due to the global financial crisis unwinding its overheated credit and real estate boom.",
    ("Estonia",       2008): "Estonia's GDP began contracting in 2008 as rising global risk aversion and a domestic credit bubble started to burst, foreshadowing the severe 2009 recession.",
    ("Estonia",       2006): "Estonia posted 9.8% growth driven by EU accession-fuelled foreign investment, a booming property market and rapid credit expansion.",
    ("Estonia",       2007): "Estonia's economy grew 7.6% as strong FDI inflows and domestic consumption continued, though overheating pressures were already building.",
    ("Estonia",       2011): "Estonia rebounded with 7.6% growth as export competitiveness, restored after painful wage cuts in 2009, powered a strong recovery.",

    # Ukraine
    ("Ukraine",       2009): "Ukraine's GDP fell 15% as the global financial crisis collapsed steel export prices (its main industry) and a banking crisis froze credit across the economy.",
    ("Ukraine",       2014): "Ukraine's GDP contracted 10% following the Euromaidan revolution, Russia's annexation of Crimea and the outbreak of conflict in the Donbas, which together triggered capital flight and trade disruption.",
    ("Ukraine",       2006): "Ukraine's GDP grew 7.6% on strong steel and chemical exports and recovering domestic consumer spending after earlier IMF-backed stabilisation.",

    # Greece sovereign debt crisis
    ("Greece",        2009): "Greece's GDP fell 4.1% as the global financial crisis exposed its chronically high deficit; the year marked the start of a decade-long economic depression.",
    ("Greece",        2010): "Greece's GDP contracted 5.7% as the country entered an EU-IMF bailout programme and began severe austerity measures, including public-sector wage cuts and tax hikes.",
    ("Greece",        2011): "Greece's economy shrank nearly 10% as its second bailout was negotiated and a private-sector debt restructuring (the largest in history) caused widespread financial disruption.",
    ("Greece",        2012): "Greece's GDP fell another 8.3% as austerity deepened, unemployment reached 27% and domestic demand collapsed under the weight of successive budget cuts.",
    ("Greece",        2013): "Greece's economy contracted a further 2.3% as austerity continued, though the pace of decline slowed as structural reforms began to stabilise public finances.",
    ("Greece",        2006): "Greece grew 6.4% during its final pre-crisis boom years, fuelled by EU structural funds, strong tourism and construction activity ahead of the post-Olympic infrastructure push.",

    # Other European financial crisis victims
    ("Iceland",       2009): "Iceland's GDP fell 8.3% after its three major banks — with liabilities ten times the country's GDP — collapsed in 2008, triggering a currency crisis and IMF bailout.",
    ("Iceland",       2007): "Iceland grew 8.7% during its final pre-crisis boom, driven by massive bank expansion, foreign capital inflows and strong aluminium smelting revenues.",
    ("Iceland",       2010): "Iceland's GDP continued to contract by 2.2% as post-banking-crisis deleveraging persisted, though capital controls and a competitive devaluation were beginning to stabilise the economy.",
    ("Finland",       2009): "Finland's GDP plunged 8.1% as the global financial crisis hammered its export-dependent economy, particularly the electronics and paper manufacturing sectors.",
    ("Finland",       2012): "Finland's GDP contracted 1.5% as the Eurozone crisis suppressed export demand and the structural decline of Nokia weighed on the technology sector.",
    ("Germany",       2009): "Germany's GDP fell 5.5% — its worst post-war recession — as global trade collapsed and its export-driven industrial base was hit by plummeting demand from trading partners.",
    ("Ireland",       2008): "Ireland's GDP contracted 4.5% as its property bubble burst and its banking system, heavily exposed to construction loans, began to require state rescue.",
    ("Ireland",       2009): "Ireland's GDP fell 5.1% as its banking crisis deepened, the government nationalised major banks and an EU-IMF bailout became necessary.",
    ("Ireland",       2014): "Ireland's GDP surged 9.4% as US multinational companies, attracted by low corporate tax rates, shifted IP assets and profits into Ireland, significantly inflating measured output.",
    ("Italy",         2009): "Italy's GDP fell 5.3% as the global financial crisis hit its export-oriented manufacturing sector and long-standing structural weaknesses — high debt and low productivity — amplified the shock.",
    ("Italy",         2012): "Italy's GDP contracted 3.1% as EU-mandated austerity measures, introduced in response to soaring sovereign bond yields, compressed domestic demand.",
    ("Italy",         2013): "Italy's GDP fell 1.8% as austerity continued but at a reduced pace; unemployment peaked above 12% and the country remained in its longest post-war recession.",
    ("Italy",         2008): "Italy's GDP contracted 1% as the global financial crisis began to bite, exposing its high public debt and sluggish export competitiveness.",
    ("Spain",         2009): "Spain's GDP fell 3.8% as a massive property bust wiped out its construction sector, unemployment surged toward 20% and its banking system required a subsequent EU bailout.",
    ("Spain",         2012): "Spain's GDP contracted 2.9% as EU austerity conditions attached to its banking-sector bailout suppressed public investment and household spending.",
    ("Spain",         2013): "Spain's GDP fell 1.4% as fiscal consolidation continued; however, a recovering tourism sector and rising exports began to signal a gradual turnaround.",
    ("Portugal",      2009): "Portugal's GDP fell 3.1% as the global financial crisis hit its trade-dependent economy and sovereign borrowing costs began to rise sharply.",
    ("Portugal",      2011): "Portugal's GDP contracted 1.7% as it entered an EU-IMF bailout programme following a spike in sovereign bond yields, triggering deep public spending cuts.",
    ("Portugal",      2012): "Portugal's GDP fell 4% as austerity measures imposed under the bailout programme severely reduced public investment and household incomes.",
    ("Cyprus",        2009): "Cyprus's GDP contracted 2% as the global financial crisis reduced tourism revenues and its banking sector, heavily exposed to Greek sovereign debt, came under pressure.",
    ("Cyprus",        2012): "Cyprus's GDP fell 3.5% as the banking system, crippled by losses on Greek bond holdings, approached collapse and depositor bail-in negotiations began.",
    ("Cyprus",        2013): "Cyprus's GDP collapsed 6.6% after an unprecedented EU bail-in imposed losses on bank depositors above €100,000 to fund a rescue, triggering a severe domestic confidence crisis.",
    ("Netherlands",   2009): "The Netherlands' GDP fell 3.7% as the global financial crisis sharply reduced trade volumes through its critical port of Rotterdam and suppressed financial services activity.",
    ("Belgium",       2009): "Belgium's GDP contracted 1.9% as the global financial crisis hit its open, trade-dependent economy and the Fortis-Dexia banking rescue strained public finances.",
    ("Austria",       2009): "Austria's GDP fell 3.6% as the global financial crisis reduced exports and its banks — heavily exposed to Central and Eastern Europe — faced elevated loan losses.",
    ("Denmark",       2009): "Denmark's GDP fell 5% as the global financial crisis burst a domestic property bubble and reduced export demand from key trading partners.",
    ("Sweden",        2009): "Sweden's GDP fell 4.3% as the global financial crisis devastated its export-dependent manufacturing and automotive sectors, though a swift fiscal stimulus helped limit the damage.",
    ("Switzerland",   2009): "Switzerland's GDP fell 2.3% as global trade collapsed and demand for Swiss machinery, chemicals and financial services dropped sharply.",
    ("Norway",        2009): "Norway's GDP contracted 1.9% as falling oil prices and reduced global demand temporarily interrupted its petroleum-driven expansion.",
    ("Luxembourg",    2009): "Luxembourg's GDP fell 3.2% as the global financial crisis sharply reduced activity in its oversized financial services sector.",
    ("Hungary",       2009): "Hungary's GDP fell 6.7% as the global financial crisis hit its export-dependent economy and a large stock of Swiss franc-denominated mortgages created a severe household debt crisis.",
    ("Hungary",       2012): "Hungary's GDP contracted 1.3% as EU-mandated fiscal consolidation and a deteriorating business environment — marked by nationalisation of private pension funds — suppressed investment.",
    ("Croatia",       2009): "Croatia's GDP fell 6.8% as the global financial crisis reduced tourism revenues, FDI and export demand, exposing its high external debt vulnerabilities.",
    ("Croatia",       2012): "Croatia's GDP contracted 2.2% during a prolonged recession driven by weak domestic demand, fiscal consolidation and falling investment ahead of EU accession.",
    ("Serbia",        2009): "Serbia's GDP contracted 3.1% as the global financial crisis reduced FDI inflows, export demand and remittances that had fuelled its pre-crisis growth.",
    ("Serbia",        2014): "Serbia's GDP fell 1.8% due to a combination of austerity measures under an IMF programme and devastating floods that damaged agricultural output and infrastructure.",
    ("Slovenia",      2012): "Slovenia's GDP fell 2.9% as a banking crisis — caused by bad loans from state-owned banks to politically-connected enterprises — required a government-funded bailout.",
    ("Slovenia",      2007): "Slovenia grew 7.1% on strong export demand, a construction boom and rising consumer spending, making it one of the fastest-growing new EU members before the crisis.",
    ("Malta",         2009): "Malta's GDP contracted 1.4% as the global financial crisis reduced tourism and manufacturing exports, its two main growth drivers.",
    ("Japan",         2008): "Japan's GDP contracted 1.2% in 2008 as the global financial crisis hit its export-led economy, with the automotive and electronics sectors particularly hard-affected.",
    ("Japan",         2009): "Japan's GDP fell 5.7% as global trade collapsed, sharply reducing demand for its exports, and deflationary pressures that had plagued the economy for a decade intensified.",
    ("United Kingdom",2009): "The UK's GDP fell 4.6% as the global financial crisis caused a severe banking sector crisis — requiring the part-nationalisation of RBS and Lloyds — and collapsed domestic demand.",
    ("Canada",        2009): "Canada's GDP contracted 2.9% as the global financial crisis reduced US demand for its exports, particularly in the automotive sector, though its banking system avoided a domestic crisis.",
    ("United States", 2009): "The US GDP fell 2.6% in the worst recession since the Great Depression, triggered by the subprime mortgage collapse, the Lehman Brothers bankruptcy and a resulting global credit freeze.",
    ("Mexico",        2009): "Mexico's GDP fell 6.3% — one of the sharpest contractions in Latin America — due to its deep trade integration with the US, the collapse of automotive exports and the simultaneous H1N1 swine flu outbreak that devastated tourism.",
    ("Chile",         2009): "Chile's GDP contracted 1.1% as falling copper prices and reduced global demand weighed on its commodity-export-dependent economy, though its strong fiscal buffers limited the damage.",
    ("Jamaica",       2009): "Jamaica's GDP fell 4.4% as the global financial crisis sharply reduced tourism revenues, remittances and bauxite export earnings, forcing an IMF debt restructuring programme.",
    ("South Africa",  2009): "South Africa's GDP contracted 1.5% — its first recession in 17 years — as the global financial crisis reduced demand for its mineral exports and suppressed domestic manufacturing.",
    ("Malaysia",      2009): "Malaysia's GDP fell 1.5% as the global financial crisis sharply reduced export demand for its electronics and palm oil, though a large fiscal stimulus package cushioned the blow.",
    ("New Zealand",   2008): "New Zealand's GDP contracted 1% as the global financial crisis reduced export demand and commodity prices, and a domestic credit crunch dampened housing and investment activity.",
    ("United Arab Emirates", 2009): "The UAE's GDP fell 5.2% as the global financial crisis triggered a real estate collapse — particularly in Dubai — and Dubai World's near-default required an Abu Dhabi bailout.",

    # ── RUSSIA ────────────────────────────────────────────────────────────────
    ("Russian Federation", 2006): "Russia grew 8.2% driven by high oil prices, rising domestic consumption and strong FDI inflows that followed political stabilisation under President Putin.",
    ("Russian Federation", 2007): "Russia's GDP expanded 8.5% as oil revenues continued to fund a public spending boom, the middle class expanded rapidly and consumer credit growth accelerated.",
    ("Russian Federation", 2009): "Russia's GDP fell 7.8% as the global financial crisis caused oil prices to plummet from their 2008 peak, capital fled the country and domestic credit markets seized up.",
    ("Russian Federation", 2005): "Russia grew 6.4% on the back of high global oil prices and a recovering domestic economy following the 1998 rouble crisis, with energy exports funding a consumption-led expansion.",

    # ── CHINA ─────────────────────────────────────────────────────────────────
    ("China",         2005): "China grew 11.5% driven by massive export expansion following WTO accession in 2001, a construction boom and strong FDI inflows as global manufacturers relocated production.",
    ("China",         2006): "China's GDP expanded 12.7% as export growth accelerated, infrastructure investment surged and urbanisation drove exceptional demand for steel, cement and energy.",
    ("China",         2007): "China grew 14.2% — its fastest rate in decades — fuelled by a global commodity super-cycle, soaring exports and a government-backed infrastructure spending programme.",
    ("China",         2008): "China's GDP slowed to 9.7% as the global financial crisis reduced export demand in the second half of the year, though a massive 4-trillion-yuan stimulus package was announced to offset the impact.",
    ("China",         2009): "China grew 9.4% — bucking the global recession — as a massive government stimulus package, worth 12.5% of GDP, fuelled infrastructure construction and domestic consumption.",
    ("China",         2010): "China expanded 10.6% as its stimulus-fuelled recovery matured, exports rebounded strongly and the global economy recovered, reinforcing its position as the world's second-largest economy.",
    ("China",         2011): "China grew 9.5% as strong domestic investment and export recovery continued, though the government began moderating credit growth to address inflation and property market overheating.",
    ("China",         2012): "China's growth eased to 7.9% as the government deliberately slowed credit expansion to rebalance the economy away from investment toward consumption, accepting lower but more sustainable growth.",
    ("China",         2013): "China grew 7.8% as the government maintained its policy of gradual economic rebalancing, with services beginning to overtake manufacturing as the primary growth driver.",
    ("China",         2014): "China's GDP expanded 7.5% — the slowest pace in 24 years — as the property market cooled, credit growth moderated and the government signalled acceptance of a 'new normal' of lower but higher-quality growth.",

    # ── INDIA ─────────────────────────────────────────────────────────────────
    ("India",         2005): "India grew 7.9% as economic liberalisation, a booming IT services sector and strong FDI inflows accelerated its emergence as a global outsourcing hub.",
    ("India",         2006): "India's GDP expanded 8.1% driven by strong domestic consumption, rising software export revenues and an infrastructure investment push.",
    ("India",         2007): "India grew 7.7% as its services sector boomed, the rupee appreciated and foreign institutional investment surged into its equity markets.",
    ("India",         2009): "India's GDP grew 7.9% despite the global financial crisis, as its relatively closed capital account limited contagion and strong domestic demand — particularly in rural areas — sustained growth.",
    ("India",         2010): "India expanded 8.5% as post-crisis recovery, strong domestic consumption and a recovery in global trade flows pushed growth back toward its pre-crisis trend.",
    ("India",         2013): "India's GDP growth slowed to 6.4% as high inflation, a current account deficit and the rupee's sharp depreciation following the US Federal Reserve's 'taper tantrum' constrained the economy.",
    ("India",         2014): "India grew 7.4% as the newly elected Modi government's reform agenda boosted investor confidence, inflation eased and the current account deficit narrowed.",

    # ── SINGAPORE ─────────────────────────────────────────────────────────────
    ("Singapore",     2005): "Singapore grew 7.4% as its open, trade-dependent economy benefited from strong global demand for electronics and financial services.",
    ("Singapore",     2006): "Singapore expanded 9% as electronics exports surged, tourism boomed ahead of the integrated resorts opening and financial services activity intensified.",
    ("Singapore",     2007): "Singapore's GDP grew 9% as the pre-crisis global boom fuelled exceptional demand for its port, financial and logistics services.",
    ("Singapore",     2010): "Singapore grew an exceptional 14.5% — its fastest rate on record — as a sharp post-crisis rebound in electronics exports coincided with the opening of two major integrated resorts.",
    ("Singapore",     2011): "Singapore's growth moderated to 6.2% as the post-crisis export rebound normalised and the Eurozone debt crisis began to dampen global trade prospects.",

    # ── AFRICA ────────────────────────────────────────────────────────────────
    ("Ghana",         2010): "Ghana grew 7.9% as expanding oil production from its newly discovered offshore Jubilee field boosted revenues and infrastructure investment accelerated.",
    ("Ghana",         2011): "Ghana's GDP surged 14% as full production began at the Jubilee oil field — one of the largest discoveries in sub-Saharan Africa — generating a resource windfall.",
    ("Ghana",         2013): "Ghana's growth eased to 7.3% as oil production from the Jubilee field began to plateau and fiscal pressures from election-related spending pushed the deficit wider.",
    ("Nigeria",       2007): "Nigeria grew 6.6% as high global oil prices expanded government revenues, enabling increased public spending on infrastructure and social programmes.",
    ("Nigeria",       2008): "Nigeria's GDP expanded 6.8% as oil prices reached record highs, generating exceptional export revenues, though political instability in the Niger Delta disrupted some production.",
    ("Nigeria",       2009): "Nigeria grew 8% despite the global financial crisis as high oil output and a rebound in agriculture — driven by improved seed varieties — kept growth resilient.",
    ("Nigeria",       2010): "Nigeria's GDP expanded 8% as strong oil revenues and a recovering non-oil sector, particularly telecommunications and retail, fuelled broad-based growth.",
    ("Nigeria",       2013): "Nigeria grew 6.7% as GDP was formally rebased to reflect the true size of its non-oil economy, confirming it as Africa's largest economy, with strong services sector growth.",
    ("Nigeria",       2014): "Nigeria's GDP growth eased to 6.3% as falling global oil prices began to reduce government revenues, creating fiscal pressures that would intensify in subsequent years.",
    ("Kenya",         2007): "Kenya grew 6.9% on the back of strong tourism, horticulture exports and a construction boom, though post-election violence at year-end would severely disrupt the economy in 2008.",
    ("Kenya",         2010): "Kenya's GDP expanded 8.1% as the economy rebounded strongly from the post-election violence of 2008 and the global financial crisis, driven by telecommunications, trade and construction.",
    ("Uganda",        2008): "Uganda grew 8.7% driven by strong agricultural output, expanding telecommunications and increasing Chinese-backed infrastructure investment.",
    ("Uganda",        2009): "Uganda's GDP grew 6.8% despite the global financial crisis, as its relatively closed economy, strong agricultural base and remittance inflows provided resilience.",
    ("Uganda",        2011): "Uganda grew 9.4% as oil exploration activity accelerated following major discoveries in the Albertine Rift, attracting significant FDI and infrastructure spending.",
    ("Botswana",      2011): "Botswana's GDP grew 6.8% as diamond export revenues recovered following the 2009 global financial crisis and the reopening of key mines boosted production.",
    ("South Africa",  2009): "South Africa's GDP contracted 1.5% — its first recession in 17 years — as the global financial crisis reduced demand for its mineral exports and suppressed domestic manufacturing.",
    ("Seychelles",    2011): "Seychelles' GDP grew 9.5% as tourism recovered strongly post-crisis and the government's IMF-backed debt restructuring programme restored investor confidence.",

    # ── MIDDLE EAST ───────────────────────────────────────────────────────────
    ("Jordan",        2005): "Jordan grew 8.2% driven by strong FDI inflows following regional stability, rising phosphate export revenues and a construction boom in Amman.",
    ("Jordan",        2008): "Jordan's GDP expanded 7.2% as high oil prices benefited Gulf trading partners who remit funds to Jordan, and phosphate export revenues surged.",
    ("Israel",        2007): "Israel grew 6.3% as its technology sector boomed, attracting record venture capital investment, and strong domestic consumption fuelled a broad economic expansion.",
    ("United Arab Emirates", 2011): "The UAE's GDP grew 6.7% as Abu Dhabi's oil revenues surged with high global prices and Dubai's economy recovered from its 2009 debt crisis, with tourism and trade expanding strongly.",

    # ── LATIN AMERICA ─────────────────────────────────────────────────────────
    ("Brazil",        2007): "Brazil grew 6.1% as high commodity prices, a domestic credit boom and rising middle-class consumption drove its strongest performance in over two decades.",
    ("Brazil",        2010): "Brazil's GDP surged 7.5% — its fastest growth in 25 years — as a commodity export boom to China, strong consumer credit expansion and pre-World Cup infrastructure spending converged.",
    ("Colombia",      2006): "Colombia grew 6.7% as President Uribe's security improvements attracted FDI, oil production expanded and domestic consumer confidence reached multi-decade highs.",
    ("Colombia",      2007): "Colombia's GDP expanded 6.7% as oil and coal exports boomed, US-Colombia trade ties strengthened and security improvements continued to unlock private investment.",
    ("Colombia",      2011): "Colombia grew 6.9% as record oil production — boosted by major new field discoveries — and a construction boom drove broad-based expansion.",
    ("Peru",          2011): "Peru grew 6.3% as high copper and gold prices, record mining investment and a domestic consumption boom positioned it as one of Latin America's star performers.",
    ("Peru",          2012): "Peru's GDP expanded 6.1% as infrastructure spending accelerated and mining output remained strong, though falling commodity prices began to moderate the growth pace.",
    ("Panama",        2011): "Panama grew 11.9% as the Panama Canal expansion project reached peak construction activity, attracting massive infrastructure investment and creating a construction and logistics boom.",
    ("Panama",        2012): "Panama's GDP expanded 9.6% as Canal expansion work continued and its financial services sector and logistics hub attracted strong regional FDI.",
    ("Uruguay",       2010): "Uruguay grew 7.8% as commodity prices boomed, FDI in agribusiness and free-trade zones surged and strong fiscal management maintained investor confidence.",
    ("Dominican Republic", 2010): "The Dominican Republic grew 8.4% as tourism rebounded post-crisis, remittance inflows recovered and infrastructure investment accelerated.",
    ("Chile",         2006): "Chile grew 6.1% on the back of record copper prices — the commodity accounts for over 50% of exports — and strong domestic investment in mining infrastructure.",
    ("Chile",         2011): "Chile grew 6.2% as copper prices hit historic highs and reconstruction investment following the 2010 earthquake boosted construction and manufacturing activity.",
    ("Chile",         2012): "Chile's GDP expanded 6.2% as copper production continued at capacity and robust domestic demand kept growth above the regional average.",

    # ── SOUTHEAST & EAST ASIA ─────────────────────────────────────────────────
    ("Indonesia",     2007): "Indonesia grew 6.3% as commodity exports boomed, domestic consumption expanded and political stability under President Yudhoyono attracted increasing FDI.",
    ("Indonesia",     2008): "Indonesia's GDP grew 6% as high commodity prices — particularly palm oil and coal — offset emerging global financial turbulence.",
    ("Indonesia",     2010): "Indonesia expanded 6.2% as a post-crisis rebound in commodity exports, rising domestic consumption and strong FDI restored its pre-crisis growth trajectory.",
    ("Indonesia",     2011): "Indonesia grew 6.2% on strong palm oil, coal and natural gas exports and a rapidly expanding domestic consumer market as its middle class grew.",
    ("Indonesia",     2012): "Indonesia's GDP grew 6% as domestic consumption — now Southeast Asia's largest economy's primary driver — remained robust despite moderating commodity prices.",
    ("Malaysia",      2007): "Malaysia grew 6.3% as electronics exports remained strong, the construction sector expanded and domestic consumption was buoyed by government spending ahead of elections.",
    ("Malaysia",      2010): "Malaysia's GDP grew 7.4% — its fastest in a decade — as a sharp post-crisis rebound in electronics exports and a government stimulus programme drove recovery.",
    ("Philippines",   2007): "The Philippines grew 6.5% as strong remittance inflows from overseas workers, BPO sector expansion and consumer spending powered broad-based growth.",
    ("Philippines",   2010): "The Philippines grew 7.3% as BPO exports boomed, remittances hit record levels and post-crisis pent-up consumer demand was released.",
    ("Philippines",   2012): "The Philippines expanded 6.9% as BPO revenue surpassed $13 billion, remittances set records and infrastructure spending accelerated under the Aquino administration.",
    ("Philippines",   2013): "The Philippines grew 6.8% despite Typhoon Hainan — the strongest ever recorded at landfall — as post-disaster reconstruction and continued BPO expansion drove resilient growth.",
    ("Philippines",   2014): "The Philippines GDP expanded 6.4% as reconstruction spending after 2013's Typhoon Yolanda continued and the BPO sector maintained its position as Asia's top outsourcing destination.",
    ("Thailand",      2010): "Thailand's GDP surged 7.5% as post-crisis export recovery — particularly in automotive and electronics — rebounded sharply despite political unrest that led to the Red Shirt protests.",
    ("Thailand",      2012): "Thailand's GDP grew 7.2% as reconstruction spending following the severe 2011 floods — which had halted major industrial estates — generated a strong bounce-back in investment.",
    ("Viet Nam",      2006): "Vietnam grew 7% as WTO accession preparation attracted record FDI, manufacturing exports boomed and domestic consumption expanded on rising incomes.",
    ("Viet Nam",      2007): "Vietnam's GDP expanded 7.1% following WTO accession, attracting a record $20.3 billion in FDI pledges and accelerating its integration into global supply chains.",
    ("Viet Nam",      2010): "Vietnam grew 6.4% as export-oriented manufacturing recovered, foreign direct investment returned and the government's stimulus programme from 2009 continued to support domestic demand.",
    ("Viet Nam",      2014): "Vietnam's GDP grew 6.4% as Samsung's massive smartphone manufacturing investment made it the country's top exporter and FDI inflows hit record levels.",
    ("Myanmar",       2012): "Myanmar's GDP expanded 7.3% as economic sanctions began to ease following democratic reforms under President Thein Sein, unlocking foreign investment and tourism.",
    ("Myanmar",       2013): "Myanmar grew 8.4% as post-sanctions economic opening accelerated — Western companies entered the market, FDI surged and infrastructure spending ramped up rapidly.",
    ("Cambodia",      2011): "Cambodia grew 7.3% as garment exports expanded strongly, tourism recovered and construction activity accelerated in Phnom Penh.",
    ("Nepal",         2008): "Nepal's GDP grew 6.1% as remittances from overseas workers — accounting for over 20% of GDP — rose strongly and post-civil-war reconstruction activity continued.",

    # ── CENTRAL ASIA & EASTERN EUROPE ─────────────────────────────────────────
    ("Azerbaijan",    2011): "Azerbaijan's GDP contracted 1.6% as oil production declined from fields in natural depletion and a sharp fall in the State Oil Fund's investment programme temporarily weighed on the economy.",
    ("Latvia",        2012): "Latvia's GDP rebounded 7.3% as its painful internal devaluation — public-sector wages cut by 25% and spending sharply reduced — restored competitiveness and exports surged.",
    ("Lithuania",     2006): "Lithuania grew 7.4% as EU structural funds, strong FDI inflows and a domestic credit and construction boom drove exceptional pre-crisis expansion.",
    ("Lithuania",     2011): "Lithuania's GDP grew 6.3% as its export-led recovery — particularly in manufacturing and timber products — was among the strongest post-crisis rebounds in the EU.",
    ("Poland",        2006): "Poland grew 6.2% as EU structural funds financed infrastructure investment, FDI surged and domestic consumption rose on improving labour market conditions.",
    ("Poland",        2007): "Poland's GDP expanded 6.8% — its strongest performance since the 1990s transition — as construction boomed ahead of Euro 2012, EU funds flowed and wages rose sharply.",
    ("Bulgaria",      2005): "Bulgaria grew 7.1% as EU accession drew near, attracting FDI, and domestic credit expansion fuelled a consumer and construction boom.",
    ("Bulgaria",      2006): "Bulgaria expanded 6.8% on the back of strong FDI, EU pre-accession spending and a domestic credit boom that drove construction and retail growth.",
    ("Bulgaria",      2007): "Bulgaria's GDP grew 6.7% as EU accession in January attracted record FDI inflows and domestic consumption expanded sharply on rising wages and credit.",
    ("Bulgaria",      2008): "Bulgaria grew 6.1% as EU structural fund transfers accelerated, though signs of overheating — rising inflation and a large current account deficit — were building.",
    ("North Macedonia", 2007): "North Macedonia grew 6.5% as EU integration progress, rising FDI and strong private consumption fuelled above-average regional growth.",
    ("Czechia",       2005): "Czechia grew 6.4% on the back of strong FDI in its automotive and electronics sectors post-EU accession, and rising export competitiveness.",
    ("Czechia",       2006): "Czechia's GDP expanded 6.6% as Volkswagen, Skoda and electronics multinationals ramped up production and EU structural funds accelerated infrastructure investment.",

    # ── SOUTH ASIA ────────────────────────────────────────────────────────────
    ("Bangladesh",    2008): "Bangladesh grew 6% despite the global financial crisis as its garment export sector — accounting for 80% of export earnings — proved resilient and remittances remained strong.",
    ("Bangladesh",    2011): "Bangladesh's GDP grew 6.5% as garment exports boomed — the sector benefited from global brands diversifying away from China — and infrastructure investment accelerated.",
    ("Bangladesh",    2012): "Bangladesh grew 6.5% as the ready-made garment sector continued to expand and remittances from overseas workers hit record levels.",
    ("Pakistan",      2006): "Pakistan's GDP grew 6.1% as strong agricultural output, rising FDI and a consumer credit boom drove its strongest economic performance since the 1990s.",

    # ── CARIBBEAN ─────────────────────────────────────────────────────────────
    ("Cayman Islands",2010): "The Cayman Islands' GDP contracted 2.7% as post-crisis deleveraging in its financial services sector — the dominant industry — continued and offshore fund flows remained subdued.",
    ("Brunei Darussalam", 2013): "Brunei's GDP contracted 2.1% as oil and gas production — accounting for 90% of government revenue — declined and no significant economic diversification offset the loss.",
}


def get_gdp_context(country: str, year: int, gdp_value: float) -> str:
    """
    Returns a one-sentence explanation of why a country's GDP moved in a given year.

    Strategy:
      1. Exact lookup in GDP_CONTEXT_TABLE  (fast, always accurate)
      2. Anthropic API call for unknowns    (dynamic, covers everything)
      3. Plain data fallback               (if API unavailable)

    Args:
        country   : country name as it appears in ventures.csv
        year      : founding year (int)
        gdp_value : gdp_growth float from WorldBank or ventures.csv

    Returns:
        str: one sentence suitable for display in the app
    """
    # ── Step 1: static table lookup ──────────────────────────────────────────
    explanation = GDP_CONTEXT_TABLE.get((country, year))
    if explanation:
        log_event("INFO", f"GDP context (static): {country} {year}")
        return explanation

    # ── Step 2: Anthropic API fallback ───────────────────────────────────────
    direction = "grew" if gdp_value >= 0 else "contracted"
    magnitude = abs(round(gdp_value, 1))
    try:
        import anthropic
        client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env
        prompt = (
            f"In exactly one sentence, explain the primary economic, political or "
            f"external reason why {country}'s GDP {direction} by {magnitude}% in {year}. "
            f"Write for an investor or entrepreneur evaluating a startup in that market. "
            f"Be specific and factual — name the actual event or driver. No preamble."
        )
        message = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=120,
            messages=[{"role": "user", "content": prompt}]
        )
        result = message.content[0].text.strip()
        log_event("INFO", f"GDP context (API): {country} {year}")
        return result

    except Exception as e:
        log_event("WARN", f"GDP context API failed for {country} {year}: {e}")

    # ── Step 3: plain data fallback ──────────────────────────────────────────
    direction_word = "grew" if gdp_value >= 0 else "contracted"
    return (
        f"{country}'s GDP {direction_word} by {abs(round(gdp_value, 1))}% in {year}; "
        f"no detailed context is available for this market at this time."
    )


print("GDP Context Engine loaded.")
print(f"Static table covers {len(GDP_CONTEXT_TABLE)} country+year events.")

# Quick smoke test
_tests = [
    ("Latvia",   2009, -16.0),
    ("China",    2007,  14.2),
    ("Kenya",    2010,   8.1),
    ("Greece",   2012,  -8.3),
    ("Singapore",2010,  14.5),
]
print()
for c, y, g in _tests:
    ctx = get_gdp_context(c, y, g)
    print(f"  [{c} {y}] {ctx[:90]}...")


GDP Context Engine loaded.
Static table covers 168 country+year events.

[2026-02-23 21:27:54] [INFO] GDP context (static): Latvia 2009
  [Latvia 2009] Latvia's GDP collapsed by 16% as the global financial crisis burst a credit-fuelled proper...
[2026-02-23 21:27:54] [INFO] GDP context (static): China 2007
  [China 2007] China grew 14.2% — its fastest rate in decades — fuelled by a global commodity super-cycle...
[2026-02-23 21:27:54] [INFO] GDP context (static): Kenya 2010
  [Kenya 2010] Kenya's GDP expanded 8.1% as the economy rebounded strongly from the post-election violenc...
[2026-02-23 21:27:54] [INFO] GDP context (static): Greece 2012
  [Greece 2012] Greece's GDP fell another 8.3% as austerity deepened, unemployment reached 27% and domesti...
[2026-02-23 21:27:54] [INFO] GDP context (static): Singapore 2010
  [Singapore 2010] Singapore grew an exceptional 14.5% — its fastest rate on record — as a sharp post-crisis ...


#### Normalization Helpers

In [9]:
def normalize(value, feature_name):
    """Scale value to 0-100 using stored min/max bounds."""
    min_val = feature_bounds_df.loc[feature_name, "min"]
    max_val = feature_bounds_df.loc[feature_name, "max"]
    if max_val == min_val:
        return 50.0
    return float(np.clip(((value - min_val) / (max_val - min_val)) * 100, 0, 100))


def normalize_inverted(value, feature_name):
    """Like normalize() but higher raw value → lower score (used for inflation)."""
    return 100 - normalize(value, feature_name)

#### Opportunity Scorer

In [10]:
# ── Weight profiles ───────────────────────────────────────────────────────────
# Two profiles grounded in the data analysis:
#
# ENTREPRENEUR — asking "Is this market fertile enough to start in?"
#   • trend_slope    0.35  strongest predictor for bootstrapped startups (corr=0.77)
#   • news_sentiment 0.25  second strongest; market mood without funding as backstop
#   • gdp_growth     0.15  entering a market — macro conditions matter more
#   • reddit_density 0.12  community interest in an idea
#   • news_volume    0.08  moderate signal
#   • startup_density 0.03 HIGH = saturated market (bad for new entrant) → inverted below
#   • log_funding    0.00  meaningless — entrepreneur has no funding yet
#   • inflation      0.02  very weak predictor; NOT inverted (high inflation = emerging
#                          market opportunity, penalising it was wrong for entrepreneurs)
#
# INVESTOR — asking "Has this company earned its score and is the market behind it?"
#   • log_funding    0.22  other smart money already validated the space
#   • trend_slope    0.20  is the market still growing?
#   • news_sentiment 0.18  third strongest signal overall
#   • news_volume    0.10  coverage = awareness
#   • reddit_density 0.10  social traction
#   • startup_density 0.08 HIGH = proven market with exit ecosystem (good for investor)
#   • gdp_growth     0.10  macro backdrop for the company's market
#   • inflation      0.02  very weak predictor for both profiles

WEIGHTS_ENTREPRENEUR = {
    "trend_slope":     0.35,
    "news_sentiment":  0.25,
    "gdp_growth":      0.15,
    "reddit_density":  0.12,
    "news_volume":     0.08,
    "startup_density": 0.03,
    "log_funding":     0.00,
    "inflation":       0.02,
}

WEIGHTS_INVESTOR = {
    "trend_slope":     0.20,
    "news_sentiment":  0.18,
    "log_funding":     0.22,
    "news_volume":     0.10,
    "reddit_density":  0.10,
    "startup_density": 0.08,
    "gdp_growth":      0.10,
    "inflation":       0.02,
}

# Keep the original as a fallback default (backwards compatible)
WEIGHTS = WEIGHTS_INVESTOR

assert abs(sum(WEIGHTS_ENTREPRENEUR.values()) - 1.0) < 1e-6, "Entrepreneur weights must sum to 1.0"
assert abs(sum(WEIGHTS_INVESTOR.values())     - 1.0) < 1e-6, "Investor weights must sum to 1.0"


def compute_opportunity_score(signals, user_type: str = "investor") -> float:
    """
    Computes a 0-100 opportunity score using weights tuned per user type.

    Parameters
    ----------
    signals   : validated signals dict (from validate_signals)
    user_type : "entrepreneur" or "investor" (default "investor")
                - entrepreneur : weights favour market momentum signals;
                                 log_funding excluded (they have none yet);
                                 startup_density INVERTED (high = saturated = bad);
                                 inflation NOT inverted (emerging markets penalised wrongly)
                - investor     : weights include log_funding; startup_density normal
                                 (high = proven market = good); inflation inverted.

    Returns
    -------
    float : score 0-100
    """
    if user_type not in ("entrepreneur", "investor"):
        log_event("WARN", f"Unknown user_type '{user_type}', defaulting to 'investor'")
        user_type = "investor"

    weights = WEIGHTS_ENTREPRENEUR if user_type == "entrepreneur" else WEIGHTS_INVESTOR
    s = dict(signals)

    # ── Impute ALL missing/NaN scored signals with dataset medians ────────────
    for feature in ["trend_slope", "gdp_growth", "inflation",
                    "news_volume", "news_sentiment", "startup_density",
                    "reddit_density"]:
        val = s.get(feature)
        if val is None or (isinstance(val, float) and (np.isnan(val) or np.isinf(val))):
            median = (
                float(feature_bounds_df.loc[feature, "median"])
                if feature in feature_bounds_df.index else 0.0
            )
            s[feature] = median
            log_event("WARN", f"Imputed missing '{feature}' with median ({median})")

    # funding_total_usd: impute from log_funding median so log transform stays valid
    raw_funding = s.get("funding_total_usd")
    if raw_funding is None or (isinstance(raw_funding, float) and (np.isnan(raw_funding) or np.isinf(raw_funding))):
        log_funding_median = float(feature_bounds_df.loc["log_funding", "median"]) \
            if "log_funding" in feature_bounds_df.index else 0.0
        raw_funding = float(np.expm1(log_funding_median))
        s["funding_total_usd"] = raw_funding
        log_event("WARN", "Imputed missing 'funding_total_usd' from log_funding median")

    s["log_funding"] = float(np.log1p(max(0, raw_funding)))

    # ── Normalise — direction differs by user type for two signals ────────────
    #
    # startup_density:
    #   investor     → normal    (high density = proven market = good)
    #   entrepreneur → inverted  (high density = saturated market = bad)
    #
    # inflation:
    #   investor     → inverted  (high inflation = macro risk = bad)
    #   entrepreneur → normal    (high inflation often = fast-growing emerging market)

    if user_type == "entrepreneur":
        startup_density_norm = normalize_inverted(s["startup_density"], "startup_density")
        inflation_norm       = normalize(s["inflation"], "inflation")
    else:
        startup_density_norm = normalize(s["startup_density"], "startup_density")
        inflation_norm       = normalize_inverted(s["inflation"], "inflation")

    normalized = {
        "trend_slope":     normalize(s["trend_slope"],      "trend_slope"),
        "gdp_growth":      normalize(s["gdp_growth"],       "gdp_growth"),
        "reddit_density":  normalize(s["reddit_density"],   "reddit_density"),
        "news_sentiment":  normalize(s["news_sentiment"],   "news_sentiment"),
        "news_volume":     normalize(s["news_volume"],      "news_volume"),
        "startup_density": startup_density_norm,
        "log_funding":     normalize(s["log_funding"],      "log_funding"),
        "inflation":       inflation_norm,
    }

    score = sum(weights[k] * normalized[k] for k in weights)

    # Final NaN guard
    if np.isnan(score) or np.isinf(score):
        log_event("WARN", "Score was NaN/Inf after normalisation — returning 50.0 (neutral)")
        return 50.0

    return round(float(score), 4)


#### Base Collector

In [11]:
class BaseCollector(ABC):
    """
    Every collector must implement collect().
    Returns a dict of signal_name → float value.
    On failure, returns an empty dict (fallback handled by AggregationEngine).
    """

    @abstractmethod
    def collect(self, industry: str, country: str, year: int) -> dict:
        pass

#### VenturesCollector (CSV Fallback)

In [12]:
class VenturesCollector(BaseCollector):
    """
    Reads from ventures.csv.
    Fuzzy-matches industry and supports year-range fallback.
    """

    def __init__(self, filepath):
        self.df = pd.read_csv(filepath)
        log_event("INFO", f"VenturesCollector loaded {len(self.df)} rows from {filepath}")

    def collect(self, industry: str, country: str, year: int) -> dict:
        # Exact match first
        filtered = self.df[
            (self.df["industry"].str.lower() == industry.lower()) &
            (self.df["country"].str.lower()  == country.lower())  &
            (self.df["founded_year"]          == year)
        ]

        # Year-range fallback (+-3 years)
        if filtered.empty:
            log_event("WARN", f"No exact match for ({industry}, {country}, {year}). Trying +-3yr range.")
            filtered = self.df[
                (self.df["industry"].str.lower() == industry.lower()) &
                (self.df["country"].str.lower()  == country.lower())  &
                (self.df["founded_year"].between(year - 3, year + 3))
            ]

        if filtered.empty:
            log_event("WARN", f"VenturesCollector: no data found for ({industry}, {country}, {year})")
            return {}

        row = filtered[[
            "trend_slope", "gdp_growth", "reddit_density",
            "news_sentiment", "news_volume", "startup_density",
            "funding_total_usd", "inflation"
        ]].median()

        return row.to_dict()

    def get_all_rows(self) -> pd.DataFrame:
        """Return full dataset for Mode A (historical batch scoring)."""
        return self.df.copy()

#### GoogleTrendsCollector

In [13]:
# Install if needed:  pip install pytrends

class GoogleTrendsCollector(BaseCollector):
    """
    Fetches trend_slope from Google Trends via pytrends.
    Uses RATE_LIMITERS['GoogleTrends'] before every API call.
    """

    COUNTRY_CODE_MAP = {
        "united states": "US", "kenya": "KE", "united kingdom": "GB",
        "india": "IN", "germany": "DE", "france": "FR", "canada": "CA",
        "australia": "AU", "nigeria": "NG", "south africa": "ZA",
    }

    def collect(self, industry: str, country: str, year: int) -> dict:
        try:
            from pytrends.request import TrendReq

            RATE_LIMITERS["GoogleTrends"].wait()   # rate limit enforced here

            geo       = self.COUNTRY_CODE_MAP.get(country.lower(), "")
            pytrends  = TrendReq(hl="en-US", tz=360, timeout=(10, 25))
            timeframe = f"{year}-01-01 {year}-12-31"
            pytrends.build_payload([industry], cat=0, timeframe=timeframe, geo=geo)

            data = pytrends.interest_over_time()

            if data.empty or industry not in data.columns:
                log_event("WARN", f"GoogleTrends: no data for '{industry}'")
                return {}

            series = data[industry].values.astype(float)
            x      = np.arange(len(series))
            if len(series) < 2:
                return {}

            slope = float(np.polyfit(x, series, 1)[0]) / 100.0
            log_event("INFO", f"GoogleTrends: trend_slope={slope:.4f}")
            return {"trend_slope": slope}

        except Exception as e:
            log_event("ERROR", f"GoogleTrendsCollector failed: {str(e)}")
            return {}

#### WorldBankCollector

In [14]:
class WorldBankCollector(BaseCollector):
    """
    Fetches gdp_growth and inflation from the World Bank Open Data API.
    No API key required. Uses RATE_LIMITERS['WorldBank'].
    """

    BASE_URL = "https://api.worldbank.org/v2/country/{iso}/indicator/{indicator}"

    COUNTRY_ISO_MAP = {
        "united states": "US", "kenya": "KE", "united kingdom": "GB",
        "india": "IN", "germany": "DE", "france": "FR", "canada": "CA",
        "australia": "AU", "nigeria": "NG", "south africa": "ZA",
        "china": "CN", "brazil": "BR", "chile": "CL", "estonia": "EE",
    }

    def _fetch_indicator(self, iso, indicator, year):
        RATE_LIMITERS["WorldBank"].wait()   # rate limit enforced here
        url    = self.BASE_URL.format(iso=iso, indicator=indicator)
        params = {"date": str(year), "format": "json", "per_page": 1}
        resp   = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        payload = resp.json()
        if len(payload) < 2 or not payload[1]:
            return None
        value = payload[1][0].get("value")
        return float(value) if value is not None else None

    def collect(self, industry: str, country: str, year: int) -> dict:
        iso = self.COUNTRY_ISO_MAP.get(country.lower())
        if not iso:
            log_event("WARN", f"WorldBankCollector: no ISO code for '{country}'")
            return {}

        result = {}
        try:
            gdp = self._fetch_indicator(iso, "NY.GDP.MKTP.KD.ZG", year)
            if gdp is not None:
                result["gdp_growth"] = gdp
                log_event("INFO", f"WorldBank: gdp_growth={gdp}")
        except Exception as e:
            log_event("ERROR", f"WorldBank GDP failed: {str(e)}")

        try:
            inf = self._fetch_indicator(iso, "FP.CPI.TOTL.ZG", year)
            if inf is not None:
                result["inflation"] = inf
                log_event("INFO", f"WorldBank: inflation={inf}")
        except Exception as e:
            log_event("ERROR", f"WorldBank Inflation failed: {str(e)}")

        return result

#### RedditCollector

In [15]:
class RedditCollector(BaseCollector):
    """
    Fetches reddit_density using the Pushshift/PullPush archive API.
    Mirrors the approach in red.py: keyword-based, year-scoped, daily density.
    Uses RATE_LIMITERS['Reddit'] before every request.

    Why PullPush instead of Reddit's live API:
    - Reddit's public /search.json ignores the query and returns trending posts,
      producing the same 100-post count regardless of keyword — fake data.
    - PullPush indexes historical Reddit submissions by keyword and timestamp,
      returning genuine keyword-specific counts for any year.
    """

    SEARCH_URL = "https://api.pullpush.io/reddit/search/submission/"
    HEADERS    = {"User-Agent": "OpportunityScoutAI/1.0"}

    # Industry → keyword list, mirroring red.py INDUSTRY_KEYWORDS
    INDUSTRY_KEYWORDS = {
        'E-Commerce & Retail':            ['ecommerce', 'online shopping', 'retail technology'],
        'Technology & Software':          ['software', 'saas', 'cloud computing'],
        'Healthcare & Life Sciences':     ['digital health', 'telemedicine', 'healthcare technology'],
        'Finance & Fintech':              ['fintech', 'digital banking', 'payment technology'],
        'Finance Technology':             ['fintech', 'digital banking', 'payment technology'],
        'Media & Entertainment':          ['streaming services', 'digital media', 'content creation'],
        'Education & Training':           ['edtech', 'online learning', 'educational technology'],
        'Real Estate & Construction':     ['proptech', 'real estate technology', 'construction technology'],
        'Travel & Hospitality':           ['travel technology', 'online booking', 'hotel technology'],
        'Food & Beverage':                ['food delivery', 'food technology', 'restaurant technology'],
        'Transportation & Logistics':     ['logistics technology', 'supply chain', 'delivery services'],
        'Marketing & Advertising':        ['digital marketing', 'martech', 'advertising technology'],
        'Internet & Web Services':        ['web services', 'internet technology', 'web applications'],
        'Data & Analytics':               ['data analytics', 'big data', 'business intelligence'],
        'Gaming':                         ['video games', 'gaming', 'esports'],
        'Social & Community':             ['social media', 'social networking', 'online community'],
        'Consumer & Lifestyle':           ['consumer technology', 'lifestyle products', 'personal technology'],
        'Manufacturing & Industrial':     ['industrial automation', 'manufacturing technology', 'industry 4.0'],
        'Sports & Recreation':            ['fitness technology', 'sports technology', 'wellness'],
        'Communications & Telecom':       ['telecommunications', '5g technology', 'telecom'],
        'Energy & Clean Tech':            ['clean energy', 'renewable energy', 'cleantech'],
        'Enterprise & Business Services': ['business software', 'enterprise solutions', 'b2b software'],
        'Security & Privacy':             ['cybersecurity', 'information security', 'data privacy'],
        'AI & Machine Learning':          ['artificial intelligence', 'machine learning', 'deep learning'],
        'Customer Service & CRM':         ['crm software', 'customer service', 'customer experience'],
        'HR & Recruiting':                ['recruitment technology', 'hr software', 'talent management'],
        'Government & Politics':          ['govtech', 'government technology', 'civic technology'],
        'Software':                       ['software', 'saas', 'cloud computing'],
        'News':                           ['news media', 'digital news', 'online journalism'],
        'Publishing':                     ['digital publishing', 'ebooks', 'online publishing'],
        'Tourism':                        ['travel technology', 'tourism technology', 'online booking'],
    }

    def _keyword_density(self, keyword: str, year: int) -> float:
        """Fetch posts for one keyword in one year and return daily density."""
        # epoch timestamps for Jan 1 and Dec 31 of target year
        start_ts = int(time.mktime(time.strptime(f"{year}-01-01", "%Y-%m-%d")))
        end_ts   = int(time.mktime(time.strptime(f"{year}-12-31", "%Y-%m-%d")))

        RATE_LIMITERS["Reddit"].wait()
        params = {
            "q":     keyword,
            "after": start_ts,
            "before": end_ts,
            "size":  100,
            "sort":  "asc",
        }
        try:
            resp = requests.get(self.SEARCH_URL, params=params,
                                headers=self.HEADERS, timeout=15)
            if resp.status_code != 200:
                log_event("WARN", f"PullPush {resp.status_code} for '{keyword}'")
                return 0.0
            data = resp.json().get("data", [])
            if len(data) < 2:
                return 0.0
            # compute daily density from first to last post timestamp
            # PullPush sometimes returns created_utc as a string — cast to int defensively
            days = (int(data[-1]["created_utc"]) - int(data[0]["created_utc"])) / 86400
            return round(len(data) / days, 4) if days > 0 else 0.0
        except Exception as e:
            log_event("ERROR", f"PullPush keyword '{keyword}': {str(e)}")
            return 0.0

    def collect(self, industry: str, country: str, year: int) -> dict:
        try:
            # Map industry name to keyword list
            keywords = self.INDUSTRY_KEYWORDS.get(industry)
            if not keywords:
                # Generic fallback: use industry name itself as a single keyword
                keywords = [industry.lower()]
                log_event("WARN", f"RedditCollector: no keyword map for '{industry}', using name as query")

            densities = [self._keyword_density(kw, year) for kw in keywords]
            avg_density = round(sum(densities) / len(densities), 4)

            log_event("INFO", f"Reddit (PullPush): '{industry}' {year} -> density={avg_density}")
            return {"reddit_density": avg_density}

        except Exception as e:
            log_event("ERROR", f"RedditCollector failed: {str(e)}")
            return {}


#### GDELTCollector

Uses **Google BigQuery** to query the GDELT dataset — the same approach used in `gdelt_test.py`
which was used to build the original ventures.csv signals.

**Why BigQuery instead of the GDELT Doc API:**
- The free GDELT Doc 2.0 REST API timed out on every live run (confirmed in notebook logs).
- BigQuery is reliable, fast, and uses the same service account key (`strong_gdelt.json`) you already have.
- The `_bq_cache` dict ensures the same industry+year is only queried once per session.

**Setup:** Place `strong_gdelt.json` in the same folder as the notebook before running.

**Signals returned:**
- `news_volume` — article count for the industry's GDELT theme code in the target year
- `news_sentiment` — GDELT AvgTone mapped from (-10,+10) to (0, 6.6) to match ventures.csv scale

**Year routing:**
- 2015+ uses `gdelt-bq.gdeltv2.gkg_partitioned` (same as gdelt_test.py Batch 1)
- 2005-2014 uses `gdelt-bq.full.events` filtered to business actors (same as Batch 2)

In [16]:
class GDELTCollector(BaseCollector):
    """
    Fetches news_volume and news_sentiment via Google BigQuery (GDELT dataset).
    Mirrors the approach in gdelt_test.py: service account auth + SQL on gdelt-bq.

    Why BigQuery instead of the GDELT Doc 2.0 REST API:
    - The free GDELT Doc API times out consistently (confirmed in live run logs).
    - BigQuery gives reliable, fast access to the same underlying data.
    - The service account key (strong_gdelt.json) authenticates automatically.

    Results are cached in _bq_cache (dict) so the same industry+year is only
    queried once per notebook session regardless of how many times collect() is called.

    news_sentiment: GDELT AvgTone mapped from (-10,+10) -> (0,6.6) to match
                    ventures.csv scale.
    """

    PROJECT_ID    = "strong-augury-487515-u7"

    # Path to the service account JSON.
    # Set this to the full absolute path if the file is not in the notebook folder.
    # e.g. CREDENTIALS = r"C:\Users\HP\projects\strong_gdelt.json"
    CREDENTIALS   = "../python_files/strong_gdelt.json"

    # Same industry->theme-code map as gdelt_test.py
    INDUSTRY_MAP = {
        'Media & Entertainment':          'GENERAL_ENTERTAINMENT',
        'Travel & Hospitality':           'TOURISM',
        'Technology & Software':          'TECHNOLOGY',
        'E-Commerce & Retail':            'ECONOMY_ECOMMERCE',
        'Real Estate & Construction':     'ECONOMY_REALESTATE',
        'Education & Training':           'EDUCATION',
        'Internet & Web Services':        'INTERNET',
        'Food & Beverage':                'FOOD_SECURITY',
        'Healthcare & Life Sciences':     'HEALTH_SERVICES',
        'Data & Analytics':               'COMPUTING_DATA_MINING',
        'Consumer & Lifestyle':           'SOCIETY_LIFESTYLE',
        'Transportation & Logistics':     'TRANSPORTATION',
        'Sports & Recreation':            'SPORTS',
        'Manufacturing & Industrial':     'MANUFACTURING',
        'Marketing & Advertising':        'ADVERTISING',
        'Gaming':                         'GAMES',
        'Finance & Fintech':              'ECONOMY_FINTECH',
        'Finance Technology':             'ECONOMY_FINTECH',
        'Communications & Telecom':       'TELECOMMUNICATIONS',
        'Social & Community':             'SOCIETY',
        'Energy & Clean Tech':            'ENERGY_RENEWABLE',
        'Enterprise & Business Services': 'ECONOMY_BUSINESS_SERVICES',
        'AI & Machine Learning':          'COMPUTING_ARTIFICIAL_INTELLIGENCE',
        'Customer Service & CRM':         'CUSTOMER_SERVICE',
        'HR & Recruiting':                'LABOR_RECRUITMENT',
        'Security & Privacy':             'SECURITY_SERVICES',
        'Government & Politics':          'GOVERNMENT',
        'Software':                       'TECHNOLOGY',
        'News':                           'GENERAL_ENTERTAINMENT',
        'Publishing':                     'GENERAL_ENTERTAINMENT',
        'Tourism':                        'TOURISM',
    }

    # In-memory cache so the same BQ query never runs twice in a session
    _bq_cache: dict = {}

    def _get_client(self):
        """
        Lazy-init BigQuery client using the service account JSON.
        Searches the notebook folder and common parent directories automatically.
        Set CREDENTIALS to a full absolute path to skip the search.
        """
        import os
        from google.cloud import bigquery

        creds_path = self.CREDENTIALS

        # If it is not a full path and the file is not found in cwd,
        # walk up two directory levels to find it (handles running from subfolders)
        if not os.path.isabs(creds_path) and not os.path.isfile(creds_path):
            for parent in [
                os.path.dirname(os.path.abspath(".")),          # one level up
                os.path.dirname(os.path.dirname(os.path.abspath("."))),  # two levels up
                os.path.abspath("."),                            # cwd (already tried)
            ]:
                candidate = os.path.join(parent, os.path.basename(creds_path))
                if os.path.isfile(candidate):
                    creds_path = candidate
                    log_event("INFO", f"GDELT: found credentials at {creds_path}")
                    break
            else:
                raise FileNotFoundError(
                    f"Credentials file '{self.CREDENTIALS}' not found.\n"
                    f"Place strong_gdelt.json in the same folder as the notebook, "
                    f"or set GDELTCollector.CREDENTIALS to its full absolute path."
                )

        os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = creds_path
        return bigquery.Client(project=self.PROJECT_ID, location="US")

    def _query_bq(self, theme_code: str, year: int) -> dict:
        """Run the BigQuery SQL and return {news_volume, news_sentiment}."""
        from google.cloud import bigquery
        client = self._get_client()

        if year >= 2015:
            # Partitioned table — efficient for 2015+
            sql = f"""
                SELECT
                    COUNT(*) as ArticleVolume,
                    ROUND(AVG(SAFE_CAST(SPLIT(V2Tone, ',')[OFFSET(0)] AS FLOAT64)), 4) as AvgSentiment
                FROM `gdelt-bq.gdeltv2.gkg_partitioned`
                WHERE _PARTITIONDATE BETWEEN '{year}-01-01' AND '{year}-12-31'
                  AND themes LIKE '%{theme_code}%'
            """
        else:
            # Historical table for 2005-2014
            sql = f"""
                SELECT
                    COUNT(*) as ArticleVolume,
                    ROUND(AVG(AvgTone), 4) as AvgSentiment
                FROM `gdelt-bq.full.events`
                WHERE Year = {year}
                  AND (Actor1Type1Code = 'BUS' OR Actor2Type1Code = 'BUS')
            """

        job    = client.query(sql)
        rows   = list(job.result())
        if not rows or rows[0].ArticleVolume == 0:
            return {}

        raw_volume    = int(rows[0].ArticleVolume)
        raw_sentiment = rows[0].AvgSentiment  # GDELT AvgTone: roughly -10 to +10

        # Map AvgTone (-10,+10) -> (0, 6.6) to match ventures.csv news_sentiment scale
        if raw_sentiment is not None:
            mapped_sentiment = round(float(np.clip((raw_sentiment + 10) * (6.6 / 20.0), 0, 6.6)), 4)
        else:
            mapped_sentiment = None

        result = {"news_volume": float(raw_volume)}
        if mapped_sentiment is not None:
            result["news_sentiment"] = mapped_sentiment
        return result

    def collect(self, industry: str, country: str, year: int) -> dict:
        try:
            theme_code = self.INDUSTRY_MAP.get(industry)
            if not theme_code:
                log_event("WARN", f"GDELTCollector: no theme code for '{industry}', skipping")
                return {}

            cache_key = f"{theme_code}|{year}"
            if cache_key in self._bq_cache:
                log_event("INFO", f"GDELT BQ cache hit: {cache_key}")
                return self._bq_cache[cache_key]

            RATE_LIMITERS["GDELT"].wait()
            result = self._query_bq(theme_code, year)

            if result:
                log_event("INFO", f"GDELT BQ: '{industry}' {year} -> volume={result.get('news_volume')}, "
                          f"sentiment={result.get('news_sentiment')}")
            else:
                log_event("WARN", f"GDELT BQ: no results for '{industry}' {year}")

            self._bq_cache[cache_key] = result
            return result

        except Exception as e:
            log_event("ERROR", f"GDELTCollector failed: {str(e)}")
            return {}


####  AggregationEngine

- `run_live()` — Mode B. Calls live collectors (rate-limited), falls back to CSV, scores, quality-checks.
- `run_historical()` — Mode A. Batch-scores all rows from ventures.csv.

In [17]:
class AggregationEngine:

    def __init__(self, ventures_path: str):
        self.ventures_collector = VenturesCollector(ventures_path)
        self.live_collectors = [
            GoogleTrendsCollector(),
            WorldBankCollector(),
            RedditCollector(),
            GDELTCollector(),
        ]
        log_event("INFO", "AggregationEngine initialized")

    # MODE B — Live scoring (web app / API)

    def run_live(self, industry: str, country: str, year: int,
                 user_type: str = "investor") -> dict:
        """
        Full live pipeline:
          1. Check cache
          2. Collect from live APIs (rate-limited)
          3. Fill gaps from ventures.csv fallback
          4. Validate signals
          5. Score opportunity  (weights depend on user_type)
          6. Score data quality
          7. GDP context explanation
          8. Return structured JSON

        Parameters
        ----------
        industry  : industry name string
        country   : country name string
        year      : target year (int)
        user_type : "entrepreneur" or "investor"
                    Controls which weight profile is used and how startup_density
                    and inflation are normalised. Passed through to the output so
                    the frontend knows which profile produced the score.

        Output fields:
          status, mode, user_type, input, score, score_profile,
          data_quality, quality_label, quality_breakdown,
          gdp_context, signals, issues
        """
        if user_type not in ("entrepreneur", "investor"):
            log_event("WARN", f"Invalid user_type '{user_type}', defaulting to 'investor'")
            user_type = "investor"

        cache_key = get_cache_key(industry, country, year) + f"|{user_type}"
        cached    = get_cached(cache_key)
        if cached:
            return cached

        # Step 1: Collect from live sources
        live_signals = {}
        for collector in self.live_collectors:
            name = collector.__class__.__name__
            try:
                data = collector.collect(industry, country, year)
                live_signals.update(data)
                log_event("INFO", f"{name} returned signals: {list(data.keys())}")
            except Exception as e:
                log_event("ERROR", f"{name} crashed unexpectedly: {str(e)}")

        # Step 2: Fill gaps with historical CSV fallback
        merged_signals = dict(live_signals)
        historical     = self.ventures_collector.collect(industry, country, year) or {}

        fallback_used = []
        for feature in REQUIRED_FEATURES:
            if feature not in merged_signals or merged_signals[feature] is None:
                fallback_val = historical.get(feature)
                if fallback_val is not None:
                    merged_signals[feature] = fallback_val
                    fallback_used.append(feature)

        if fallback_used:
            log_event("INFO", f"Fallback used for: {fallback_used}")

        # Step 3: Validate
        validated, issues = validate_signals(merged_signals)

        # Step 4: Score opportunity using the correct weight profile
        score = compute_opportunity_score(validated, user_type=user_type)

        # Step 5: Score data quality
        quality_info = score_data_quality(validated, live_signals)

        # Step 6: GDP context
        gdp_val     = validated.get("gdp_growth", 0) or 0
        gdp_context = get_gdp_context(country, year, gdp_val)

        # Step 7: Describe which profile was used (useful for the frontend tooltip)
        if user_type == "entrepreneur":
            score_profile = (
                "Entrepreneur profile: weighted toward market momentum (trend, sentiment). "
                "Funding excluded. High competition penalised."
            )
        else:
            score_profile = (
                "Investor profile: weighted toward funding traction and market validation. "
                "High startup density treated as a positive signal."
            )

        # Step 8: Build output
        result = {
            "status":            "success",
            "mode":              "live",
            "user_type":         user_type,
            "score_profile":     score_profile,
            "input": {
                "industry":  industry,
                "country":   country,
                "year":      year,
            },
            "score":             score,
            "data_quality":      quality_info["data_quality"],
            "quality_label":     quality_info["quality_label"],
            "quality_breakdown": quality_info["quality_breakdown"],
            "gdp_context":       gdp_context,
            "signals":           validated,
            "issues":            issues,
        }

        set_cache(cache_key, result)
        return result

    # MODE A — Historical batch scoring (ML team training data)

    def run_historical(self, sample_n: int = None,
                       user_type: str = "investor") -> pd.DataFrame:
        """
        Batch-scores all (or sample_n) rows from ventures.csv.
        Returns DataFrame with original columns + pipeline_score +
        data_quality + gdp_context + user_type.

        Parameters
        ----------
        sample_n  : number of rows to sample (None = all rows)
        user_type : "entrepreneur" or "investor" — selects weight profile
        """
        source_df = self.ventures_collector.get_all_rows()

        if sample_n:
            source_df = source_df.sample(n=min(sample_n, len(source_df)), random_state=42)

        log_event("INFO", f"run_historical: scoring {len(source_df)} rows (user_type='{user_type}')...")
        scores   = []
        quality  = []
        contexts = []

        for _, row in source_df.iterrows():
            signals = {
                "trend_slope":       row.get("trend_slope"),
                "gdp_growth":        row.get("gdp_growth"),
                "reddit_density":    row.get("reddit_density"),
                "news_sentiment":    row.get("news_sentiment"),
                "news_volume":       row.get("news_volume"),
                "startup_density":   row.get("startup_density"),
                "funding_total_usd": row.get("funding_total_usd"),
                "inflation":         row.get("inflation"),
            }
            validated, _ = validate_signals(signals)
            score        = compute_opportunity_score(validated, user_type=user_type)
            q_info       = score_data_quality(validated, live_signals={})
            gdp_val      = validated.get("gdp_growth", 0) or 0
            context      = get_gdp_context(
                country   = str(row.get("country", "")),
                year      = int(row.get("founded_year", 0)),
                gdp_value = float(gdp_val),
            )

            scores.append(score)
            quality.append(q_info["data_quality"])
            contexts.append(context)

        source_df                    = source_df.copy()
        source_df["pipeline_score"]  = scores
        source_df["data_quality"]    = quality
        source_df["gdp_context"]     = contexts
        source_df["user_type"]       = user_type

        log_event("INFO", f"run_historical complete. Mean pipeline_score={np.mean(scores):.2f}")
        return source_df


#### Unit Tests

In [18]:
class TestPipeline(unittest.TestCase):

    def setUp(self):
        self.ventures_path = "../Data/ventures.csv"
        self.engine = AggregationEngine(self.ventures_path)

    def _full_signals(self):
        return {
            "trend_slope": 0.1, "gdp_growth": 2.5, "reddit_density": 5.0,
            "news_sentiment": 6.0, "news_volume": 150000.0,
            "startup_density": 4.0, "funding_total_usd": 500000.0, "inflation": 2.0
        }

    def test_validate_signals_valid(self):
        validated, issues = validate_signals(self._full_signals())
        self.assertEqual(issues, {})
        self.assertIn("trend_slope", validated)

    def test_validate_signals_missing_key(self):
        _, issues = validate_signals({"trend_slope": 0.1})
        self.assertIn("gdp_growth", issues)
        self.assertEqual(issues["gdp_growth"], "missing_key")

    def test_validate_signals_nan(self):
        sigs = self._full_signals()
        sigs["trend_slope"] = float("nan")
        _, issues = validate_signals(sigs)
        self.assertEqual(issues["trend_slope"], "nan_value")

    def test_scorer_range(self):
        validated, _ = validate_signals(self._full_signals())
        score = compute_opportunity_score(validated)
        self.assertGreaterEqual(score, 0)
        self.assertLessEqual(score, 100)

    def test_ventures_no_match(self):
        collector = VenturesCollector(self.ventures_path)
        result = collector.collect("NonexistentIndustryXYZ", "Mars", 1800)
        self.assertEqual(result, {})

    def test_live_mode_field(self):
        result = self.engine.run_live("Finance Technology", "United States", 2010)
        self.assertEqual(result["mode"], "live")

    def test_historical_returns_dataframe(self):
        df_result = self.engine.run_historical(sample_n=10)
        self.assertIsInstance(df_result, pd.DataFrame)
        self.assertIn("pipeline_score", df_result.columns)
        self.assertEqual(len(df_result), 10)

    def test_data_quality_all_live(self):
        sigs = self._full_signals()
        validated, _ = validate_signals(sigs)
        q = score_data_quality(validated, live_signals=validated)
        self.assertEqual(q["quality_label"], "High")
        self.assertGreaterEqual(q["data_quality"], 0.75)

    def test_data_quality_all_fallback(self):
        sigs = self._full_signals()
        validated, _ = validate_signals(sigs)
        q = score_data_quality(validated, live_signals={})
        self.assertEqual(q["quality_label"], "Medium")

    def test_rate_limiter_does_not_block_short_burst(self):
        limiter = RateLimiter(max_calls=5, period_seconds=60)
        start = time.monotonic()
        for _ in range(5):
            limiter.wait()
        elapsed = time.monotonic() - start
        self.assertLess(elapsed, 1.0)

    def test_output_has_quality_fields(self):
        result = self.engine.run_live("Finance Technology", "United States", 2010)
        self.assertIn("data_quality",      result)
        self.assertIn("quality_label",     result)
        self.assertIn("quality_breakdown", result)


suite  = unittest.TestLoader().loadTestsFromTestCase(TestPipeline)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

test_data_quality_all_fallback (__main__.TestPipeline) ... ok
test_data_quality_all_live (__main__.TestPipeline) ... ok
test_historical_returns_dataframe (__main__.TestPipeline) ... 

[2026-02-23 21:27:55] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-23 21:27:55] [INFO] AggregationEngine initialized
[2026-02-23 21:27:55] [INFO] Data quality: 0.5 (Medium) | breakdown: {'trend_slope': 'fallback', 'gdp_growth': 'fallback', 'inflation': 'fallback', 'reddit_density': 'fallback', 'news_volume': 'fallback', 'news_sentiment': 'fallback'}
[2026-02-23 21:27:55] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-23 21:27:55] [INFO] AggregationEngine initialized
[2026-02-23 21:27:55] [INFO] Data quality: 1.0 (High) | breakdown: {'trend_slope': 'live', 'gdp_growth': 'live', 'inflation': 'live', 'reddit_density': 'live', 'news_volume': 'live', 'news_sentiment': 'live'}


ok
test_live_mode_field (__main__.TestPipeline) ... 

[2026-02-23 21:27:55] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-23 21:27:55] [INFO] AggregationEngine initialized
[2026-02-23 21:27:55] [INFO] run_historical: scoring 10 rows (user_type='investor')...
[2026-02-23 21:27:55] [INFO] Data quality: 0.5 (Medium) | breakdown: {'trend_slope': 'fallback', 'gdp_growth': 'fallback', 'inflation': 'fallback', 'reddit_density': 'fallback', 'news_volume': 'fallback', 'news_sentiment': 'fallback'}
[2026-02-23 21:27:55] [WARN] GDP context API failed for Israel 2012: No module named 'anthropic'
[2026-02-23 21:27:55] [INFO] Data quality: 0.5 (Medium) | breakdown: {'trend_slope': 'fallback', 'gdp_growth': 'fallback', 'inflation': 'fallback', 'reddit_density': 'fallback', 'news_volume': 'fallback', 'news_sentiment': 'fallback'}
[2026-02-23 21:27:55] [INFO] GDP context (static): United States 2009
[2026-02-23 21:27:55] [INFO] Data quality: 0.5 (Medium) | breakdown: {'trend_slope': 'fallback', 'gdp_growth': 'fallback', 'in

C:\Users\HP\anaconda3\envs\learn-env\lib\site-packages\google\api_core\_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.8.5). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
C:\Users\HP\anaconda3\envs\learn-env\lib\site-packages\google\auth\__init__.py:52: FutureWarning: You are using a Python version 3.8 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.8"), FutureWarning)
C:\Users\HP\anaconda3\envs\learn-env\lib\site-packages\google\oauth2\__init__.py:38: FutureWarning: You are using a Python version 3.8 past its end of life. Google will update google-auth with c

...

[2026-02-23 21:28:17] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-23 21:28:17] [INFO] AggregationEngine initialized
[2026-02-23 21:28:17] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-23 21:28:17] [INFO] AggregationEngine initialized
[2026-02-23 21:28:17] [WARN] Signal issues detected: {'gdp_growth': 'missing_key', 'reddit_density': 'missing_key', 'news_sentiment': 'missing_key', 'news_volume': 'missing_key', 'startup_density': 'missing_key', 'funding_total_usd': 'missing_key', 'inflation': 'missing_key'}


ok
test_validate_signals_valid (__main__.TestPipeline) ... ok
test_ventures_no_match (__main__.TestPipeline) ... 

[2026-02-23 21:28:17] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-23 21:28:17] [INFO] AggregationEngine initialized
[2026-02-23 21:28:17] [WARN] Signal issues detected: {'trend_slope': 'nan_value'}
[2026-02-23 21:28:18] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-23 21:28:18] [INFO] AggregationEngine initialized
[2026-02-23 21:28:18] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-23 21:28:18] [INFO] AggregationEngine initialized
[2026-02-23 21:28:18] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-23 21:28:18] [WARN] No exact match for (NonexistentIndustryXYZ, Mars, 1800). Trying +-3yr range.
[2026-02-23 21:28:18] [WARN] VenturesCollector: no data found for (NonexistentIndustryXYZ, Mars, 1800)


ok

----------------------------------------------------------------------
Ran 11 tests in 23.339s

OK


<unittest.runner.TextTestResult run=11 errors=0 failures=0>

#### Live Test

Tests the full Mode B pipeline end-to-end with real API calls.
Check `quality_breakdown` to see exactly which signals came from live collectors
vs the CSV fallback. If a collector is blocked or unavailable the fallback handles it automatically.

In [20]:
print("=" * 60)
print("LIVE PIPELINE TEST — both user profiles")
print("=" * 60)

clear_cache()

engine = AggregationEngine("../Data/ventures.csv")

TEST_INDUSTRY = "Finance Technology"
TEST_COUNTRY  = "United States"
TEST_YEAR     = 2015

for profile in ("entrepreneur", "investor"):
    print(f"\n--- Profile: {profile.upper()} ---")
    result = engine.run_live(TEST_INDUSTRY, TEST_COUNTRY, TEST_YEAR, user_type=profile)

    print(f"  Status         : {result['status']}")
    print(f"  Score          : {result['score']}")
    print(f"  Score Profile  : {result['score_profile']}")
    print(f"  Data Quality   : {result['data_quality']:.4f} ({result['quality_label']})")
    print(f"  GDP Context    : {result['gdp_context']}")
    print(f"\n  Quality Breakdown:")
    for feat, status in result['quality_breakdown'].items():
        tag = "[OK]" if status == "live" else "[~~]" if status == "fallback" else "[!!]"
        print(f"    {feat:<24} -> {tag}  {status}")
    print(f"\n  Signals:")
    for k, v in result['signals'].items():
        print(f"    {k:<24}: {v}")

print("\n" + "=" * 60)
print("Score difference (investor - entrepreneur):", 
      round(engine.run_live(TEST_INDUSTRY, TEST_COUNTRY, TEST_YEAR, user_type="investor")['score']
          - engine.run_live(TEST_INDUSTRY, TEST_COUNTRY, TEST_YEAR, user_type="entrepreneur")['score'], 4))

# Assertions — both profiles must produce valid scores
for profile in ("entrepreneur", "investor"):
    r = engine.run_live(TEST_INDUSTRY, TEST_COUNTRY, TEST_YEAR, user_type=profile)
    assert r["status"] == "success",              f"FAIL [{profile}]: status not success"
    assert 0 <= r["score"] <= 100,                f"FAIL [{profile}]: score out of range"
    assert r["user_type"] == profile,             f"FAIL [{profile}]: user_type mismatch"
    assert "gdp_context" in r,                    f"FAIL [{profile}]: missing gdp_context"
    assert "score_profile" in r,                  f"FAIL [{profile}]: missing score_profile"

print("\nAll assertions passed.")


LIVE PIPELINE TEST — both user profiles
[2026-02-23 21:34:34] [INFO] Cache cleared
[2026-02-23 21:34:34] [INFO] VenturesCollector loaded 25018 rows from ../Data/ventures.csv
[2026-02-23 21:34:34] [INFO] AggregationEngine initialized

--- Profile: ENTREPRENEUR ---
[2026-02-23 21:34:40] [INFO] GoogleTrends: trend_slope=0.0003
[2026-02-23 21:34:40] [INFO] GoogleTrendsCollector returned signals: ['trend_slope']
[2026-02-23 21:34:41] [INFO] WorldBank: gdp_growth=2.94555045484053
[2026-02-23 21:34:42] [INFO] WorldBank: inflation=0.118627135552451
[2026-02-23 21:34:42] [INFO] WorldBankCollector returned signals: ['gdp_growth', 'inflation']
[2026-02-23 21:34:49] [INFO] Reddit (PullPush): 'Finance Technology' 2015 -> density=2.7815
[2026-02-23 21:34:49] [INFO] RedditCollector returned signals: ['reddit_density']
[2026-02-23 21:34:51] [ERROR] GDELTCollector failed: 403 Quota exceeded: Your project exceeded quota for free query bytes scanned. For more information, see https://cloud.google.com/big

#### Fallback Test

Proves the safety net works end-to-end.

We deliberately pass an industry that **no live collector will meaningfully match** AND
a year that is outside ventures.csv's date range so the CSV fallback also finds nothing.

**Expected behaviour:**
- GoogleTrends returns `{}` (unknown industry or rate-limited)
- WorldBank **may** return live gdp_growth and inflation for United States/1995 (it covers historical data)
- RedditCollector returns near-zero density (no posts about basket weaving)
- GDELTCollector returns `{}` (no theme code mapped for this industry)
- VenturesCollector finds no exact or range match, returns `{}`
- Missing signals (trend_slope, news_sentiment, news_volume, funding) are **imputed from medians**
- Pipeline returns `status: success` with a valid numeric score — does NOT crash
- `quality_label` will be **Low or Medium** depending on how many WorldBank signals resolved
  (if WorldBank returned gdp+inflation live → Medium; if not → Low)

**Score is no longer NaN:** the expanded imputation in `compute_opportunity_score` now
covers all eight scored signals, so a missing trend_slope or funding can never poison the result.

In [21]:
print("=" * 60)
print("FALLBACK TEST  (bad input — no live data, no CSV match)")
print("=" * 60)

clear_cache()

fallback_result = engine.run_live(
    industry="Underwater Basket Weaving",  # will never match any live collector
    country="United States",
    year=1995                              # outside ventures.csv date range (2005-2018)
)

print(f"\n  Status         : {fallback_result['status']}")
print(f"  Mode           : {fallback_result['mode']}")
print(f"  Score          : {fallback_result['score']}   <- scored via median imputation")
print(f"  Data Quality   : {fallback_result['data_quality']} ({fallback_result['quality_label']})")

print("\n  Quality Breakdown:")
for feature, source in fallback_result["quality_breakdown"].items():
    if source == "live":
        tag = "[OK]  live"
    elif source == "fallback":
        tag = "[~~]  fallback"
    else:
        tag = "[!!]  missing"
    print(f"    {feature:<22} -> {tag}")

print("\n  Issues:")
if fallback_result["issues"]:
    for k, v in fallback_result["issues"].items():
        print(f"    {k}: {v}")
else:
    print("    none (medians imputed silently - score is low-confidence)")

# Confirm pipeline did not crash and still returned valid structure
assert fallback_result["status"] == "success",  "FAIL: pipeline crashed on bad input"
assert fallback_result["mode"]   == "live",      "FAIL: mode field missing"
assert 0 <= fallback_result["score"] <= 100,     "FAIL: score out of range"
assert fallback_result["quality_label"] in ("Low", "Medium"), "FAIL: expected Low or Medium quality label when most signals are fallback/imputed"

print("\nAll assertions passed. Fallback + imputation worked correctly.")

FALLBACK TEST  (bad input — no live data, no CSV match)
[2026-02-23 21:36:29] [INFO] Cache cleared
[2026-02-23 21:36:34] [ERROR] GoogleTrendsCollector failed: The request failed: Google returned a response with code 429
[2026-02-23 21:36:34] [INFO] GoogleTrendsCollector returned signals: []
[2026-02-23 21:36:35] [INFO] WorldBank: gdp_growth=2.6844307317787
[2026-02-23 21:36:37] [INFO] WorldBank: inflation=2.80541968853662
[2026-02-23 21:36:37] [INFO] WorldBankCollector returned signals: ['gdp_growth', 'inflation']
[2026-02-23 21:36:37] [WARN] RedditCollector: no keyword map for 'Underwater Basket Weaving', using name as query
[2026-02-23 21:36:37] [INFO] Reddit (PullPush): 'Underwater Basket Weaving' 1995 -> density=0.0
[2026-02-23 21:36:37] [INFO] RedditCollector returned signals: ['reddit_density']
[2026-02-23 21:36:37] [WARN] GDELTCollector: no theme code for 'Underwater Basket Weaving', skipping
[2026-02-23 21:36:37] [INFO] GDELTCollector returned signals: []
[2026-02-23 21:36:37] 

#### Mode A Demo: Historical Batch Scoring for ML Team

In [ ]:
scored_df = engine.run_historical(sample_n=100)

scored_df.to_csv("historical_scored.csv", index=False)
print("Saved: historical_scored.csv")

print("\nCorrelation — pipeline_score vs ground-truth opportunity_score:")
print(scored_df[["opportunity_score", "pipeline_score"]].corr())
print()
print(scored_df[["opportunity_score", "pipeline_score", "data_quality"]].describe())

---

# 🤝 Objective 2 Integration — LightGBM Model Bridge

This section connects **Objective 1 (Data Pipeline)** to **Objective 2 (ML Model)**.

## How the two notebooks relate

| What she built | What it needs | What we give it |
|---|---|---|
| `VentureFeatureEngineer` | 8 raw signals + industry + country + year + funding | `result['signals']` from `run_live()` |
| `predict_opportunity()` | A row with the raw signals filled in | We build that row from the pipeline output |
| `calibrate_score()` | A raw float prediction | Returned automatically by `predict_opportunity()` |
| `ventures_lightgbm.pkl` | 80 engineered features in the exact training order | `MLBridge.predict()` handles all 80 |

## What the pipeline supplies

Every call to `run_live()` returns `result['signals']` which contains:
`trend_slope`, `gdp_growth`, `reddit_density`, `news_sentiment`, `news_volume`,
`startup_density`, `funding_total_usd`, `inflation`

All 8 of these are **direct inputs** to the model's 80-feature schema — no unit conversion needed.
The remaining 72 features are derived (log transforms, polynomial terms, interactions,
one-hot encoding) and are computed inside `VentureFeatureEngineer.transform()`.

## One thing to know: `industry` vs `category`

The pipeline uses **`industry`** (e.g. `'Finance Technology'`) as the user-facing label.
The model was trained on **`category`** (e.g. `'Finance & Fintech'`) — the 26-bucket grouping
from `feature_engineer.ipynb`. The `MLBridge` class maps between them automatically.

## The `status` field

The model has `status_acquired`, `status_closed`, `status_operating` as features.
For a live prediction (new venture), we default to `status = 'operating'`.
The backend can let the user override this if needed.

## `funding_rounds` field

The pipeline does not collect `funding_rounds` from live APIs (it is company-specific, not
market-level). We default to `1` (seed round) for new predictions. The backend can expose
this as an optional user input.


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Objective 2 Integration — MLBridge
# Connects the pipeline (Objective 1) to the LightGBM model (Objective 2).
#
# Prerequisites:
#   pip install lightgbm joblib
#   ventures_lightgbm.pkl  must be reachable at ML_MODEL_PATH below
#   venture_features.csv   must be reachable at VENTURE_FEATURES_PATH below
# ════════════════════════════════════════════════════════════════════════════

import joblib
import re

# ── Paths — adjust if your folder layout is different ────────────────────────
ML_MODEL_PATH        = "../python_files/ventures_lightgbm.pkl"
VENTURE_FEATURES_PATH = "../Data/venture_features.csv"


# ── industry (pipeline label) → category (model label) map ──────────────────
# The pipeline uses the user-facing industry name.
# The model was trained on the 26-bucket 'category' column from ventures.csv.
# Any industry not in this map falls back to the closest category via keyword.
INDUSTRY_TO_CATEGORY = {
    # Direct matches
    'Finance Technology':             'Finance & Fintech',
    'Finance & Fintech':              'Finance & Fintech',
    'Technology & Software':          'Technology & Software',
    'AI & Machine Learning':          'AI & Machine Learning',
    'E-Commerce & Retail':            'E-Commerce & Retail',
    'Healthcare & Life Sciences':     'Healthcare & Life Sciences',
    'Media & Entertainment':          'Media & Entertainment',
    'Education & Training':           'Education & Training',
    'Real Estate & Construction':     'Real Estate & Construction',
    'Travel & Hospitality':           'Travel & Hospitality',
    'Food & Beverage':                'Food & Beverage',
    'Transportation & Logistics':     'Transportation & Logistics',
    'Sports & Recreation':            'Sports & Recreation',
    'Manufacturing & Industrial':     'Manufacturing & Industrial',
    'Marketing & Advertising':        'Marketing & Advertising',
    'Gaming':                         'Gaming',
    'Communications & Telecom':       'Communications & Telecom',
    'Social & Community':             'Social & Community',
    'Energy & Clean Tech':            'Energy & Clean Tech',
    'Enterprise & Business Services': 'Enterprise & Business Services',
    'Customer Service & CRM':         'Customer Service & CRM',
    'HR & Recruiting':                'HR & Recruiting',
    'Security & Privacy':             'Security & Privacy',
    'Government & Politics':          'Government & Politics',
    'Internet & Web Services':        'Internet & Web Services',
    'Data & Analytics':               'Data & Analytics',
    # Common pipeline industry labels that need remapping
    'Software':                       'Technology & Software',
    'News':                           'Media & Entertainment',
    'Publishing':                     'Media & Entertainment',
    'Tourism':                        'Travel & Hospitality',
    'Fintech':                        'Finance & Fintech',
    'Biotech':                        'Healthcare & Life Sciences',
    'Cleantech':                      'Energy & Clean Tech',
}

# Countries in the model's top-10 (everything else becomes 'Other')
_MODEL_TOP_COUNTRIES = {
    'Canada', 'China', 'France', 'Germany',
    'India', 'Israel', 'Singapore', 'Spain',
    'United Kingdom', 'United States'
}


class MLBridge:
    """
    Connects the Objective 1 pipeline to the Objective 2 LightGBM model.

    Usage
    -----
    bridge = MLBridge(ML_MODEL_PATH, VENTURE_FEATURES_PATH)

    # Option A: full pipeline run (live signals + ML score)
    pipeline_result = engine.run_live('Finance Technology', 'Kenya', 2020)
    ml_result       = bridge.predict_from_pipeline(pipeline_result)

    # Option B: direct call with known signals
    ml_result = bridge.predict(
        industry='Finance Technology', country='Kenya', year=2020,
        signals={'trend_slope': 0.12, 'gdp_growth': 5.1, ...},
        funding_total_usd=2_500_000
    )

    Output dict (from calibrate_score):
        score      : float 0-100
        category   : 'High Potential' | 'Medium Potential' | 'Lower Potential'
        confidence : float (0.75 / 0.85 / 0.90)
        range      : (low, high) margin of error tuple
    """

    def __init__(self, model_path: str, features_csv_path: str):
        # Load the trained model + metadata saved by modeling.ipynb
        artifacts            = joblib.load(model_path)
        self._model          = artifacts['model']
        self._feature_names  = artifacts['feature_names']   # the 80 training features
        self._model_name     = artifacts['model_name']

        # Load the feature-engineered dataset so VentureFeatureEngineer
        # can borrow training-time statistics (means, std) for z-scores.
        # We compute them once here rather than inside every transform call.
        ref_df = pd.read_csv(features_csv_path)
        self._ref_stats = {
            col: {'mean': ref_df[col].mean(), 'std': ref_df[col].std()}
            for col in ['trend_slope', 'gdp_growth', 'reddit_density']
            if col in ref_df.columns
        }

        log_event("INFO", f"MLBridge loaded '{self._model_name}' with {len(self._feature_names)} features")

    # ── public interface ──────────────────────────────────────────────────────

    def predict_from_pipeline(self, pipeline_result: dict,
                               funding_total_usd: float = None,
                               funding_rounds: int = 1,
                               status: str = 'operating') -> dict:
        """
        Takes the dict returned by engine.run_live() and produces an ML score.

        Parameters
        ----------
        pipeline_result   : the full dict from engine.run_live()
        funding_total_usd : override funding amount (USD). If None, uses the
                            value from pipeline_result['signals'] (CSV fallback).
        funding_rounds    : number of funding rounds (default 1 = seed)
        status            : 'operating' | 'acquired' | 'closed' (default 'operating')
        """
        inp   = pipeline_result['input']
        sigs  = pipeline_result['signals']

        # Use provided funding override, or fall back to pipeline signal
        funding = funding_total_usd if funding_total_usd is not None \
                  else sigs.get('funding_total_usd', 0)

        return self.predict(
            industry          = inp['industry'],
            country           = inp['country'],
            year              = inp['year'],
            signals           = sigs,
            funding_total_usd = funding,
            funding_rounds    = funding_rounds,
            status            = status,
        )

    def predict(self, industry: str, country: str, year: int,
                signals: dict,
                funding_total_usd: float = 0,
                funding_rounds: int = 1,
                status: str = 'operating') -> dict:
        """
        Build the 80-feature row from raw signals and run the model.
        """
        # Map industry → category (26-bucket model label)
        category = INDUSTRY_TO_CATEGORY.get(industry)
        if category is None:
            # keyword fallback: find best category match
            category = self._infer_category(industry)
            log_event("WARN", f"MLBridge: '{industry}' not in map, inferred '{category}'")

        # Resolve country to model's top-10 or 'Other'
        country_grouped = country if country in _MODEL_TOP_COUNTRIES else 'Other'

        # Build raw input row
        raw = {
            'founded_year':       year,
            'funding_total_usd':  float(funding_total_usd),
            'funding_rounds':     float(funding_rounds),
            'trend_slope':        float(signals.get('trend_slope',    0)),
            'news_volume':        float(signals.get('news_volume',     0)),
            'news_sentiment':     float(signals.get('news_sentiment',  0)),
            'reddit_density':     float(signals.get('reddit_density',  0)),
            'gdp_growth':         float(signals.get('gdp_growth',      0)),
            'inflation':          float(signals.get('inflation',       0)),
            'startup_density':    float(signals.get('startup_density', 0)),
            'category':           category,
            'status':             status,
            'country_grouped':    country_grouped,
        }

        # Engineer all 80 features — mirrors feature_engineer.ipynb exactly
        row = self._engineer_features(raw)

        # Align to the exact 80-column training schema
        X = self._align_features(row)

        # Predict and calibrate
        raw_score = float(self._model.predict(X)[0])
        result    = self._calibrate(raw_score)

        log_event("INFO",
            f"MLBridge: '{industry}' / {country} / {year} -> "
            f"raw={raw_score:.2f}, calibrated={result['score']} ({result['category']})"
        )
        return result

    # ── private helpers ───────────────────────────────────────────────────────

    def _engineer_features(self, raw: dict) -> dict:
        """
        Reproduces every transformation from feature_engineer.ipynb Cell 14-21.
        Input : raw dict with the base signals.
        Output: flat dict with all 80 engineered features.
        """
        d = dict(raw)
        f = d['funding_total_usd']
        t = d['trend_slope']
        g = d['gdp_growth']
        r = d['reddit_density']
        ns = d['news_sentiment']
        nv = d['news_volume']
        inf = d['inflation']
        sd = d['startup_density']
        fr = d['funding_rounds']
        yr = d['founded_year']

        # Cell 14: log transforms (feature_engineer uses log10 for log_funding
        # but log1p for funding_log — both are preserved exactly)
        d['log_funding']        = np.log10(f + 1)          # Cell 10 (used in plot)
        d['funding_log']        = np.log1p(f)              # Cell 14
        d['news_volume_log']    = np.log1p(nv)             # Cell 14
        d['reddit_density_log'] = np.log1p(r)              # Cell 14

        # Cell 16: polynomial features
        d['trend_slope_sq'] = t ** 2
        d['gdp_growth_sq']  = g ** 2
        d['funding_sqrt']   = np.sqrt(max(0, f))

        # Cell 17: interaction features
        d['trend_x_gdp']        = t * g
        d['trend_x_sentiment']  = t * ns
        d['reddit_x_sentiment'] = r * ns
        d['funding_x_rounds']   = d['funding_log'] * fr
        d['trend_x_reddit']     = t * d['reddit_density_log']
        d['gdp_x_density']      = g * sd
        d['news_x_sentiment']   = d['news_volume_log'] * ns
        d['econ_health']        = g - inf

        # Cell 18: ratio features
        d['funding_per_round']    = f / (fr + 1)
        d['real_gdp_growth']      = g / (1 + inf / 100)
        d['investment_climate']   = g / (inf + 1)
        d['ecosystem_strength']   = sd * g / 100
        d['market_validation']    = (t * 0.4
                                     + (r / 100) * 0.3
                                     + (ns / 10) * 0.3)

        # Cell 19: time features
        d['company_age']    = 2020 - yr
        d['pre_2010_flag']  = int(yr < 2010)
        d['post_2015_flag'] = int(yr >= 2015)

        # Cell 20: z-scores (using training-set statistics from venture_features.csv)
        for feat in ['trend_slope', 'gdp_growth', 'reddit_density']:
            key = feat + '_zscore'
            stats = self._ref_stats.get(feat, {'mean': 0, 'std': 1})
            std   = stats['std'] if stats['std'] != 0 else 1
            d[key] = (d[feat] - stats['mean']) / std

        # Cell 21: one-hot encoding — category (26 sectors)
        all_categories = [
            'AI & Machine Learning', 'Communications & Telecom', 'Consumer & Lifestyle',
            'Customer Service & CRM', 'Data & Analytics', 'E-Commerce & Retail',
            'Education & Training', 'Energy & Clean Tech', 'Enterprise & Business Services',
            'Finance & Fintech', 'Food & Beverage', 'Gaming', 'Government & Politics',
            'HR & Recruiting', 'Healthcare & Life Sciences', 'Internet & Web Services',
            'Manufacturing & Industrial', 'Marketing & Advertising', 'Media & Entertainment',
            'Real Estate & Construction', 'Security & Privacy', 'Social & Community',
            'Sports & Recreation', 'Technology & Software', 'Transportation & Logistics',
            'Travel & Hospitality'
        ]
        for cat in all_categories:
            d[f'cat_{cat}'] = int(d.get('category') == cat)

        # one-hot encoding — status (3 values)
        for s in ['acquired', 'closed', 'operating']:
            d[f'status_{s}'] = int(d.get('status') == s)

        # one-hot encoding — country (top 10 + Other)
        all_countries = [
            'Canada', 'China', 'France', 'Germany', 'India',
            'Israel', 'Other', 'Singapore', 'Spain', 'United Kingdom', 'United States'
        ]
        cg = d.get('country_grouped', 'Other')
        for c in all_countries:
            d[f'country_{c}'] = int(cg == c)

        # funding tiers
        if f == 0:
            tier = 'unfunded'
        elif f <= 2_000_000:
            tier = 'seed'
        elif f <= 10_000_000:
            tier = 'series_a'
        else:
            tier = 'series_b_plus'
        for t_name in ['unfunded', 'seed', 'series_a', 'series_b_plus']:
            d[f'tier_{t_name}'] = int(tier == t_name)

        return d

    def _align_features(self, engineered: dict) -> pd.DataFrame:
        """
        Selects and orders the exact 80 features the model was trained on.
        Applies the same column-name cleaning as modeling.ipynb Cell 6.
        Missing features are filled with 0.
        """
        # Clean column names the same way modeling.ipynb does
        cleaned_map = {
            re.sub(r'[\[\]<>{},:"]', '_', k): v
            for k, v in engineered.items()
        }
        row = {feat: cleaned_map.get(feat, 0.0) for feat in self._feature_names}
        return pd.DataFrame([row])[self._feature_names].fillna(0).astype(float)

    def _infer_category(self, industry: str) -> str:
        """
        Keyword fallback for unmapped industry names.
        Returns the closest category or 'Technology & Software' as last resort.
        """
        il = industry.lower()
        kw_map = [
            ('finance', 'Finance & Fintech'),
            ('health', 'Healthcare & Life Sciences'),
            ('medical', 'Healthcare & Life Sciences'),
            ('education', 'Education & Training'),
            ('media', 'Media & Entertainment'),
            ('entertainment', 'Media & Entertainment'),
            ('retail', 'E-Commerce & Retail'),
            ('ecommerce', 'E-Commerce & Retail'),
            ('travel', 'Travel & Hospitality'),
            ('food', 'Food & Beverage'),
            ('energy', 'Energy & Clean Tech'),
            ('security', 'Security & Privacy'),
            ('government', 'Government & Politics'),
            ('gaming', 'Gaming'),
            ('game', 'Gaming'),
            ('social', 'Social & Community'),
            ('ai', 'AI & Machine Learning'),
            ('machine learning', 'AI & Machine Learning'),
            ('data', 'Data & Analytics'),
            ('transport', 'Transportation & Logistics'),
            ('logistics', 'Transportation & Logistics'),
            ('real estate', 'Real Estate & Construction'),
            ('manufacturing', 'Manufacturing & Industrial'),
            ('marketing', 'Marketing & Advertising'),
            ('hr', 'HR & Recruiting'),
            ('recruit', 'HR & Recruiting'),
            ('crm', 'Customer Service & CRM'),
            ('customer', 'Customer Service & CRM'),
            ('telecom', 'Communications & Telecom'),
            ('communication', 'Communications & Telecom'),
            ('internet', 'Internet & Web Services'),
            ('web', 'Internet & Web Services'),
            ('enterprise', 'Enterprise & Business Services'),
            ('consumer', 'Consumer & Lifestyle'),
            ('sports', 'Sports & Recreation'),
        ]
        for kw, cat in kw_map:
            if kw in il:
                return cat
        return 'Technology & Software'  # safe default

    @staticmethod
    def _calibrate(raw_score: float) -> dict:
        """
        Mirrors calibrate_score() from modeling.ipynb Cell 11 exactly.
        """
        score = float(np.clip(raw_score, 0, 100))
        if score >= 70:
            category, confidence = 'High Potential',   0.90
        elif score >= 40:
            category, confidence = 'Medium Potential', 0.85
        else:
            category, confidence = 'Lower Potential',  0.75
        return {
            'score':      round(score, 2),
            'category':   category,
            'confidence': confidence,
            'range':      (round(score - 4, 2), round(score + 4, 2)),
        }


# ── Initialise the bridge (run once at startup) ───────────────────────────────
try:
    bridge = MLBridge(ML_MODEL_PATH, VENTURE_FEATURES_PATH)
    log_event("INFO", "MLBridge ready")
except FileNotFoundError as e:
    log_event("WARN", f"MLBridge could not load: {e}. Update ML_MODEL_PATH above.")
    bridge = None


#### Integration Demo

Shows both ways to call the bridge:
- **Option A** (normal app flow): run the pipeline first, pass the result to the bridge
- **Option B** (direct call): call the bridge directly with known signals


In [ ]:
if bridge is None:
    print("MLBridge not loaded — check ML_MODEL_PATH in the cell above.")
else:
    clear_cache()
    print("=" * 60)
    print("OPTION A — Pipeline result → ML score (both profiles)")
    print("=" * 60)

    for profile in ("entrepreneur", "investor"):
        pipeline_result = engine.run_live(
            industry  = "Finance Technology",
            country   = "Kenya",
            year      = 2019,
            user_type = profile,
        )
        ml_result = bridge.predict_from_pipeline(
            pipeline_result,
            funding_total_usd = 2_500_000,
            funding_rounds    = 2,
            status            = "operating",
        )
        print(f"\n  [{profile.upper()}]")
        print(f"    Pipeline score : {pipeline_result['score']}")
        print(f"    ML score       : {ml_result['score']}  ({ml_result['category']})")
        print(f"    Confidence     : {ml_result['confidence']*100:.0f}%")
        print(f"    Score profile  : {pipeline_result['score_profile']}")
        print(f"    GDP context    : {pipeline_result['gdp_context']}")

    print("\n" + "=" * 60)
    print("OPTION B — Direct call with known signals")
    print("=" * 60)

    direct_result = bridge.predict(
        industry          = "AI & Machine Learning",
        country           = "United States",
        year              = 2020,
        funding_total_usd = 10_000_000,
        funding_rounds    = 3,
        signals = {
            "trend_slope":     0.15,
            "gdp_growth":      2.3,
            "inflation":       1.8,
            "reddit_density":  5.2,
            "news_volume":     320000,
            "news_sentiment":  6.1,
            "startup_density": 8.4,
        }
    )
    print(f"\n  ML score  : {direct_result['score']}")
    print(f"  Category  : {direct_result['category']}")
    print(f"  Confidence: {direct_result['confidence']*100:.0f}%")
    print(f"  Range     : {direct_result['range']}")

    print("\n" + "=" * 60)
    print("COMBINED OUTPUT — what the backend API returns to the frontend")
    print("=" * 60)
    pipeline_result = engine.run_live("Finance Technology", "Kenya", 2019, user_type="investor")
    ml_result       = bridge.predict_from_pipeline(pipeline_result, funding_total_usd=2_500_000)
    combined = {
        "status":            pipeline_result["status"],
        "mode":              pipeline_result["mode"],
        "user_type":         pipeline_result["user_type"],
        "score_profile":     pipeline_result["score_profile"],
        "input":             pipeline_result["input"],
        "pipeline_score":    pipeline_result["score"],
        "ml_score":          ml_result["score"],
        "ml_category":       ml_result["category"],
        "ml_confidence":     ml_result["confidence"],
        "ml_range":          ml_result["range"],
        "data_quality":      pipeline_result["data_quality"],
        "quality_label":     pipeline_result["quality_label"],
        "quality_breakdown": pipeline_result["quality_breakdown"],
        "gdp_context":       pipeline_result["gdp_context"],
        "signals":           pipeline_result["signals"],
    }
    for k, v in combined.items():
        print(f"  {k:<22}: {v}")


---

# FastAPI Integration Guide

This section documents how the backend engineer connects this pipeline to the FastAPI layer.
The pipeline is the **data and scoring engine** — FastAPI is the **transport layer** around it.
They are kept completely separate so the pipeline can be tested and run independently.

---

## How the Integration Works

The backend engineer imports `AggregationEngine` and calls `run_live()` inside a FastAPI route.
The output dict from `run_live()` maps directly to the API response model with no transformation.

```
FastAPI POST /score   ->  AggregationEngine.run_live()        ->  JSON response
FastAPI GET  /batch   ->  AggregationEngine.run_historical()  ->  CSV or JSON
```

---

## Step-by-Step for the Backend Engineer

### 1. Install dependencies
```bash
pip install fastapi uvicorn pytrends requests
```

### 2. Recommended project structure
```
opportunity_scout/
├── data_pipeline/
│   ├── pipeline.py            <- export this notebook as Python (remove demo cells)
│   └── feature_bounds.csv     <- generated by Cell 3, must be present at runtime
├── api/
│   ├── main.py                <- FastAPI app
│   └── models.py              <- Pydantic request/response models
└── Data/
    └── ventures.csv
```

### 3. Convert the notebook to a module
Export this notebook as `pipeline.py` via File -> Download as -> Python.
Delete Cells 17-20 (tests and demos) from the exported file.
The module just needs the class definitions + `feature_bounds_df` loaded at import time.

### 4. The FastAPI app  (api/main.py)
```python
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from data_pipeline.pipeline import AggregationEngine

app    = FastAPI(title="Opportunity Scout AI", version="1.0")
engine = AggregationEngine("Data/ventures.csv")  # instantiated ONCE at startup

class ScoreRequest(BaseModel):
    industry : str
    country  : str
    year     : int

class ScoreResponse(BaseModel):
    status            : str
    mode              : str     # always 'live'
    input             : dict
    score             : float
    data_quality      : float
    quality_label     : str     # 'High' | 'Medium' | 'Low'
    quality_breakdown : dict
    signals           : dict
    issues            : dict

@app.post("/score", response_model=ScoreResponse)
def score_opportunity(request: ScoreRequest):
    try:
        return engine.run_live(
            industry=request.industry,
            country=request.country,
            year=request.year
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
def health():
    return {"status": "ok"}
```

### 5. Run locally
```bash
uvicorn api.main:app --reload
```

### 6. Example curl request
```bash
curl -X POST http://localhost:8000/score \
     -H 'Content-Type: application/json' \
     -d '{"industry": "Finance Technology", "country": "United States", "year": 2024}'
```

### 7. Example response
```json
{
  "status": "success",
  "mode": "live",
  "input": {"industry": "Finance Technology", "country": "United States", "year": 2024},
  "score": 64.32,
  "data_quality": 0.833,
  "quality_label": "High",
  "quality_breakdown": {
    "trend_slope": "live",
    "gdp_growth": "live",
    "inflation": "live",
    "reddit_density": "fallback",
    "news_volume": "live",
    "news_sentiment": "live"
  },
  "signals": { "..." },
  "issues": {}
}
```

---

## What Each Team Uses From This Pipeline

| Team              | What they call                     | What they get                                      |
|-------------------|------------------------------------|----------------------------------------------------|  
| Backend / API     | `engine.run_live()`                | Full JSON -> wrap directly in `/score` endpoint    |
| ML Engineer       | `engine.run_historical()`          | DataFrame -> `historical_scored.csv` for training  |
| Explanation/SHAP  | `result['signals']`                | Feature dict -> compute SHAP values from it        |
| Frontend          | `result['score']`, `quality_label` | Score gauge + quality badge in the UI              |
| Public API        | Same `/score` endpoint             | Externally documented version of the same JSON     |

---

## Important Notes for the Backend Engineer

- `AggregationEngine` must be instantiated **once at app startup**, not per request. Instantiating it per request would reload ventures.csv on every single call.
- The in-memory `CACHE` dict works fine in development. For production, swap it with a Redis client using the same `get_cached` / `set_cache` interface — the rest of the code does not need to change.
- The `RATE_LIMITERS` are module-level singletons and persist automatically across all requests in the same process. No extra wiring needed.
- If a live collector fails (network down, API blocked), the pipeline falls back silently and still returns HTTP 200 with a lower `data_quality` score. No 500 errors are thrown by the pipeline itself — only the FastAPI wrapper should raise HTTPException.
- The `/what-if` endpoint can be built by calling `run_live()` with a modified input dict (e.g. change the year or industry) and comparing the two scores.
- The `/compare` endpoint can be built by calling `run_live()` twice with different inputs and returning both results side by side.